Update step 2.2
- embedding and save in chromadb
      - paragraph aware chunking
      - two chunking methods
- dense retrieval
  

# chroma db connect

In [2]:
%pip install chromadb sentence-transformers

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 13.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 13.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 13.7 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 12.4 MB/s eta 0:00:00
  Created wheel for pypika: filename=pypika-0.48.9-py2.py3-none-any.whl size=53801 sha256=b875f039fe92f2b18c7c1b28421a3eb952acc64a2cc6c5e16a567a2ffc70f7f8
  Stored in directory: /Users/dd/Library/Caches/pip/wheels/d5/3d/69/8d68d249cd3de2584f226e27fd431d6344f7d70fd856ebd01b
Successfully built pypika

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m 

# chunking method
ensure metadata is saved in chromadb

In [30]:
import chromadb
import json
from pathlib import Path

def check_original_metadata_in_chromadb(collection_name: str):
    """Check if original JSON metadata made it to ChromaDB."""
    try:
        client = chromadb.PersistentClient(path="./chroma_db")
        collection = client.get_collection(name=collection_name)
        
        # Get sample chunks
        sample = collection.get(limit=5, include=['metadatas'])
        
        print(f"🔍 ORIGINAL METADATA CHECK: {collection_name}")
        print("="*50)
        
        # Check what metadata fields exist
        all_keys = set()
        for metadata in sample['metadatas']:
            all_keys.update(metadata.keys())
        
        print(f"📊 Available metadata fields:")
        for key in sorted(all_keys):
            print(f"  • {key}")
        
        # Check for hybrid retrieval fields
        hybrid_fields = ['ner', 'rag_metadata', 'keywords', 'entities', 'topics', 'categories']
        found_hybrid = [field for field in hybrid_fields if field in all_keys]
        missing_hybrid = [field for field in hybrid_fields if field not in all_keys]
        
        print(f"\n🔍 Hybrid Retrieval Fields:")
        if found_hybrid:
            print(f"  ✅ Found: {found_hybrid}")
        if missing_hybrid:
            print(f"  ❌ Missing: {missing_hybrid}")
        
        # Show sample metadata
        print(f"\n📝 Sample metadata:")
        for i, metadata in enumerate(sample['metadatas'][:2]):
            print(f"\nChunk {i+1}:")
            for key, value in metadata.items():
                if isinstance(value, (list, dict)):
                    print(f"  {key}: {type(value).__name__} with {len(value) if hasattr(value, '__len__') else 'N/A'} items")
                else:
                    print(f"  {key}: {str(value)[:100]}{'...' if len(str(value)) > 100 else ''}")
        
        return all_keys
        
    except Exception as e:
        print(f"❌ Error checking {collection_name}: {e}")
        return set()

def check_original_json_structure(json_directory: str):
    """Check what metadata was in the original JSON files."""
    print(f"\n🔍 ORIGINAL JSON STRUCTURE")
    print("="*30)
    
    json_dir = Path(json_directory)
    json_files = list(json_dir.glob("*.json"))[:3]  # Check first 3 files
    
    if not json_files:
        print(f"❌ No JSON files found in {json_directory}")
        return
    
    all_original_keys = set()
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            print(f"\n📄 {json_file.name}:")
            all_original_keys.update(data.keys())
            
            for key, value in data.items():
                if isinstance(value, list):
                    if value and isinstance(value[0], dict):
                        print(f"  {key}: List of {len(value)} dicts")
                    else:
                        print(f"  {key}: List of {len(value)} items")
                elif isinstance(value, dict):
                    print(f"  {key}: Dict with {len(value)} keys")
                else:
                    print(f"  {key}: {type(value).__name__}")
        
        except Exception as e:
            print(f"❌ Error reading {json_file}: {e}")
    
    print(f"\n📊 All original fields found:")
    for key in sorted(all_original_keys):
        print(f"  • {key}")
    
    return all_original_keys

def compare_metadata_preservation(json_directory: str, collection_name: str):
    """Compare original JSON metadata vs ChromaDB metadata."""
    print(f"\n🔄 METADATA PRESERVATION ANALYSIS")
    print("="*40)
    
    # Get original metadata structure
    original_keys = check_original_json_structure(json_directory)
    
    # Get ChromaDB metadata structure  
    chromadb_keys = check_original_metadata_in_chromadb(collection_name)
    
    # Compare
    preserved = original_keys.intersection(chromadb_keys)
    lost = original_keys - chromadb_keys
    added = chromadb_keys - original_keys
    
    print(f"\n📊 PRESERVATION SUMMARY:")
    print(f"  ✅ Preserved: {len(preserved)} fields")
    if preserved:
        print(f"     {sorted(preserved)}")
    
    print(f"  ❌ Lost: {len(lost)} fields")  
    if lost:
        print(f"     {sorted(lost)}")
    
    print(f"  ➕ Added: {len(added)} fields")
    if added:
        print(f"     {sorted(added)}")

# Usage
if __name__ == "__main__":
    # Check what's in ChromaDB
    check_original_metadata_in_chromadb("news_structure_chunks")
    
    # Check original JSON structure
    json_directory = "/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2"
    
    # Compare preservation
    compare_metadata_preservation(json_directory, "news_structure_chunks")

🔍 ORIGINAL METADATA CHECK: news_structure_chunks
📊 Available metadata fields:
  • chunk_index
  • chunk_length
  • chunk_method
  • date
  • filename
  • language
  • original_paragraph_length
  • paragraph_heading
  • paragraph_index
  • source_path
  • title

🔍 Hybrid Retrieval Fields:
  ❌ Missing: ['ner', 'rag_metadata', 'keywords', 'entities', 'topics', 'categories']

📝 Sample metadata:

Chunk 1:
  title: Sunbathing Meteoroids
  original_paragraph_length: 2256
  filename: sunbathing-meteoroids.md
  chunk_method: document_structure
  paragraph_heading: Samples from Omani–Swiss project
  date: 2017-03
  source_path: en_news_events/2017/03/sunbathing-meteoroids.html
  chunk_length: 480
  chunk_index: 0
  paragraph_index: 0
  language: en

Chunk 2:
  chunk_length: 238
  chunk_method: document_structure
  original_paragraph_length: 2256
  source_path: en_news_events/2017/03/sunbathing-meteoroids.html
  chunk_index: 1
  paragraph_heading: Samples from Omani–Swiss project
  language: en
 

## different emebedding model supporting multilingual retrieval

### Script

🌍 Multilingual RAG Components Overview

1. **Multilingual Embedding Models**

| Model                         | Description                                                        |
|------------------------------|--------------------------------------------------------------------|
| **`multilingual-mpnet`**     | 🔹 Best overall for English/German (🇬🇧/🇩🇪) — **recommended**       |
| **`bge-m3`**                  | 🌐 Strong multilingual performance across many languages            |
| **`multilingual-minilm`**    | ⚡ Fast and lightweight, suitable for low-resource environments     |
| **`distiluse-multilingual`** | ⚖️ Good balance between speed and accuracy                         |
| **OpenAI `text-embedding-3-small`** | ☁️ High-quality, requires API key and internet access     |

---

2. **Language Filtering**
- ✅ Supports filtering by language using a `language_filter` argument
- 📌 Useful for precision in multilingual corpora

---

3. **Multilingual Search Function**

🔁 Search in both languages:
This strategy acts as an **input expansion layer before the retrieval phase** in the RAG pipeline:

```python
search_multilingual("collection_name", "Who was ETH president?", n_results=10)
search_multilingual("collection_name", "Wer war der ETH Präsident?", language_filter="de")
```

**Flow**: 

User query

   ↓
   
Multilingual reformulations (EN, DE)

   ↓
   
Separate retrievals per language

   ↓
   
Combine or re-rank retrieved documents

   ↓
   
Pass to generator (e.g., GPT) → Answer


🎯 Benefits:

✅ True multilingual support - Same embedding space for EN/DE

✅ Cross-lingual retrieval - German query finds English docs

✅ Hybrid retrieval ready - Combines with metadata filtering

In [44]:
import json
import os
import uuid
from pathlib import Path
from typing import List, Dict, Any
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions

# Langchain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

def get_recursive_splitter(chunk_size: int, chunk_overlap: int):
    """Get recursive character text splitter."""
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
        length_function=len,
    )

def get_embedding_models():
    """Initialize embedding models for semantic chunking."""
    model_kwargs = {"device": "cpu"}  # Change to "cuda" if GPU available
    
    embedding_models = {
        "mini": HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2", 
            model_kwargs=model_kwargs
        ),
        "bge-m3": HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3", 
            model_kwargs=model_kwargs
        ),
        "gte": HuggingFaceEmbeddings(
            model_name="Alibaba-NLP/gte-base-en-v1.5", 
            model_kwargs=model_kwargs
        ),
    }
    return embedding_models

def chunk_paragraph_recursive(paragraph: str, chunk_size: int = 256, chunk_overlap: int = 64) -> List[str]:
    """Chunk a single paragraph using recursive text splitter."""
    if not paragraph or len(paragraph.strip()) == 0:
        return []
    
    splitter = get_recursive_splitter(chunk_size, chunk_overlap)
    chunks = splitter.split_text(paragraph)
    return [chunk.strip() for chunk in chunks if chunk.strip()]

def chunk_paragraph_semantic(paragraph: str, model_name: str = "mini") -> List[str]:
    """Chunk a single paragraph using semantic chunking."""
    if not paragraph or len(paragraph.strip()) == 0:
        return []
    
    embedding_models = get_embedding_models()
    if model_name not in embedding_models:
        print(f"Model {model_name} not found, using 'mini' instead")
        model_name = "mini"
    
    semantic_chunker = SemanticChunker(
        embeddings=embedding_models[model_name],
        breakpoint_threshold_type="percentile"
    )
    
    try:
        chunks = semantic_chunker.split_text(paragraph)
        return [chunk.strip() for chunk in chunks if chunk.strip()]
    except Exception as e:
        print(f"Semantic chunking failed, falling back to recursive: {e}")
        return chunk_paragraph_recursive(paragraph)

def chunk_paragraph_document_structure(paragraph: str) -> List[str]:
    """Chunk based on document structure (sentences, periods)."""
    if not paragraph or len(paragraph.strip()) == 0:
        return []
    
    # Split by sentences (periods followed by space and capital letter)
    import re
    sentences = re.split(r'(?<=\.)\s+(?=[A-Z])', paragraph)
    
    # Group sentences into chunks (max 2-3 sentences per chunk)
    chunks = []
    current_chunk = ""
    sentence_count = 0
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
            
        if sentence_count == 0:
            current_chunk = sentence
        else:
            current_chunk += " " + sentence
        
        sentence_count += 1
        
        # Create chunk after 2-3 sentences or if getting too long
        if sentence_count >= 2 and (sentence_count >= 3 or len(current_chunk) > 200):
            chunks.append(current_chunk.strip())
            current_chunk = ""
            sentence_count = 0
    
    # Add remaining text
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    
    return chunks

def chunk_paragraph_hybrid(paragraph: str, chunk_size: int = 256, chunk_overlap: int = 64, 
                          semantic_model: str = "mini") -> List[str]:
    """Hybrid approach: try semantic first, fallback to recursive if needed."""
    if not paragraph or len(paragraph.strip()) == 0:
        return []
    
    # Try semantic chunking first
    try:
        semantic_chunks = chunk_paragraph_semantic(paragraph, semantic_model)
        
        # If semantic chunks are too large, apply recursive chunking to each
        final_chunks = []
        for chunk in semantic_chunks:
            if len(chunk) > chunk_size * 1.5:  # If chunk is too large
                sub_chunks = chunk_paragraph_recursive(chunk, chunk_size, chunk_overlap)
                final_chunks.extend(sub_chunks)
            else:
                final_chunks.append(chunk)
        
        return final_chunks
    
    except Exception as e:
        print(f"Hybrid chunking failed, using recursive: {e}")
        return chunk_paragraph_recursive(paragraph, chunk_size, chunk_overlap)

def process_json_file(file_path: str, chunking_method: str = "recursive", **kwargs) -> List[Dict[str, Any]]:
    """Process a single JSON file and return chunks with metadata."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        chunks_data = []
        paragraphs = data.get('paragraphs', [])
        
        # Extract original metadata for hybrid retrieval
        original_ner = data.get('ner', [])
        original_rag_metadata = data.get('rag_metadata', [])
        original_keywords = data.get('keywords', [])
        original_entities = data.get('entities', [])
        original_topics = data.get('topics', [])
        original_categories = data.get('categories', [])
        
        # Convert complex metadata to strings for ChromaDB compatibility
        def safe_metadata_value(value):
            """Convert metadata value to ChromaDB-compatible format."""
            if isinstance(value, (list, dict)):
                return json.dumps(value) if value else ""
            return str(value) if value is not None else ""
        
        # Extract keywords/entities as comma-separated strings for easier filtering
        keywords_str = ""
        entities_str = ""
        
        if isinstance(original_keywords, list):
            keywords_str = ", ".join([str(k) for k in original_keywords if k])
        elif original_keywords:
            keywords_str = str(original_keywords)
        
        if isinstance(original_entities, list):
            entities_str = ", ".join([str(e) for e in original_entities if e])
        elif original_entities:
            entities_str = str(original_entities)
        
        # Extract NER entities as searchable text
        ner_text = ""
        if isinstance(original_ner, list):
            ner_entities = []
            for ner_item in original_ner:
                if isinstance(ner_item, dict):
                    if 'text' in ner_item:
                        ner_entities.append(str(ner_item['text']))
                    elif 'entity' in ner_item:
                        ner_entities.append(str(ner_item['entity']))
                else:
                    ner_entities.append(str(ner_item))
            ner_text = ", ".join(ner_entities)
        
        for para_idx, paragraph in enumerate(paragraphs):
            # Handle new structure where paragraphs are dictionaries
            if isinstance(paragraph, dict):
                heading = paragraph.get('heading', '')
                text = paragraph.get('text', '')
                
                # Combine heading and text for chunking
                full_text = f"{heading}\n{text}" if heading else text
                paragraph_heading = heading
            else:
                # Handle old structure where paragraphs are strings
                full_text = paragraph
                paragraph_heading = ""
            
            if not full_text or len(full_text.strip()) == 0:
                continue
            
            # Apply selected chunking method
            if chunking_method == "recursive":
                chunks = chunk_paragraph_recursive(
                    full_text, 
                    kwargs.get('chunk_size', 256), 
                    kwargs.get('chunk_overlap', 64)
                )
            elif chunking_method == "semantic":
                chunks = chunk_paragraph_semantic(
                    full_text, 
                    kwargs.get('model_name', 'mini')
                )
            elif chunking_method == "document_structure":
                chunks = chunk_paragraph_document_structure(full_text)
            elif chunking_method == "hybrid":
                chunks = chunk_paragraph_hybrid(
                    full_text,
                    kwargs.get('chunk_size', 256),
                    kwargs.get('chunk_overlap', 64),
                    kwargs.get('semantic_model', 'mini')
                )
            else:
                print(f"Unknown chunking method: {chunking_method}, using recursive")
                chunks = chunk_paragraph_recursive(full_text)
            
            # Create chunk metadata with original fields preserved
            for chunk_idx, chunk_text in enumerate(chunks):
                chunk_data = {
                    "id": str(uuid.uuid4()),
                    "text": chunk_text,
                    "metadata": {
                        # Basic document metadata
                        "filename": data.get('filename', ''),
                        "title": data.get('title', ''),
                        "language": data.get('language', ''),
                        "date": data.get('date', ''),
                        "source_path": data.get('source_path', ''),
                        
                        # Chunk-specific metadata
                        "paragraph_index": para_idx,
                        "paragraph_heading": paragraph_heading,
                        "chunk_index": chunk_idx,
                        "chunk_method": chunking_method,
                        "original_paragraph_length": len(full_text),
                        "chunk_length": len(chunk_text),
                        
                        # Original metadata for hybrid retrieval
                        "ner": safe_metadata_value(original_ner),
                        "rag_metadata": safe_metadata_value(original_rag_metadata),
                        "keywords": keywords_str,
                        "entities": entities_str,
                        "topics": safe_metadata_value(original_topics),
                        "categories": safe_metadata_value(original_categories),
                        
                        # Searchable NER entities
                        "ner_entities": ner_text,
                    }
                }
                chunks_data.append(chunk_data)
        
        return chunks_data
    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return []

def save_to_chromadb(chunks_data: List[Dict[str, Any]], collection_name: str = "news_chunks", 
                    embedding_model: str = "multilingual-mpnet"):
    """Save chunks to ChromaDB with multilingual embeddings."""
    try:
        # Initialize ChromaDB client
        client = chromadb.PersistentClient(path="./chroma_db")
        
        # Get multilingual embedding function
        embedding_function = get_multilingual_embedding_function(embedding_model)
        
        # Create or get collection with custom embedding function
        try:
            collection = client.get_collection(name=collection_name)
            print(f"Using existing collection: {collection_name}")
        except:
            collection = client.create_collection(
                name=collection_name,
                embedding_function=embedding_function
            )
            print(f"Created new collection: {collection_name} with {embedding_model} embeddings")
        
        # Prepare data for ChromaDB
        documents = [chunk["text"] for chunk in chunks_data]
        metadatas = [chunk["metadata"] for chunk in chunks_data]
        ids = [chunk["id"] for chunk in chunks_data]
        
        # Add language detection to metadata
        for i, (doc, meta) in enumerate(zip(documents, metadatas)):
            if not meta.get('language'):  # Only detect if not already specified
                detected_lang = detect_language(doc)
                metadatas[i]['detected_language'] = detected_lang
        
        print(f"Generating multilingual embeddings for {len(documents)} chunks...")
        
        # Add to collection in batches (smaller batches for embedding generation)
        batch_size = 50  # Smaller batches for embedding models
        total_saved = 0
        
        for i in range(0, len(documents), batch_size):
            batch_docs = documents[i:i+batch_size]
            batch_metas = metadatas[i:i+batch_size]
            batch_ids = ids[i:i+batch_size]
            
            try:
                collection.add(
                    documents=batch_docs,
                    metadatas=batch_metas,
                    ids=batch_ids
                )
                total_saved += len(batch_docs)
                
                # Progress update
                if (i // batch_size + 1) % 10 == 0:
                    print(f"   Processed {total_saved}/{len(documents)} chunks")
                    
            except Exception as e:
                print(f"Error in batch {i//batch_size + 1}: {e}")
                # Try individual items if batch fails
                for doc, meta, chunk_id in zip(batch_docs, batch_metas, batch_ids):
                    try:
                        collection.add(
                            documents=[doc],
                            metadatas=[meta], 
                            ids=[chunk_id]
                        )
                        total_saved += 1
                    except:
                        print(f"Failed to save individual chunk: {chunk_id}")
        
        print(f"Successfully saved {total_saved} chunks with {embedding_model} embeddings")
        return True
    
    except Exception as e:
        print(f"Error saving to ChromaDB: {e}")
        return False

def process_directory(directory_path: str, chunking_method: str = "recursive", 
                     collection_name: str = "news_chunks", **kwargs):
    """Process all JSON files in a directory and save to ChromaDB."""
    directory = Path(directory_path)
    json_files = list(directory.rglob("*.json"))
    
    if not json_files:
        print(f"No JSON files found in {directory_path}")
        return
    
    print(f"Found {len(json_files)} JSON files")
    print(f"Using chunking method: {chunking_method}")
    
    all_chunks = []
    
    for json_file in json_files:
        print(f"Processing: {json_file.name}")
        chunks = process_json_file(str(json_file), chunking_method, **kwargs)
        all_chunks.extend(chunks)
        print(f"  Generated {len(chunks)} chunks")
    
    print(f"\nTotal chunks generated: {len(all_chunks)}")
    
    # Save to ChromaDB
    if all_chunks:
        save_to_chromadb(all_chunks, collection_name)
    else:
        print("No chunks to save")

# Usage examples
if __name__ == "__main__":
    json_directory = "/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2"

    
    # Example 1: Recursive chunking
    #process_directory(
    #    directory_path=json_directory,
    #    chunking_method="recursive",
    #    collection_name="news_recursive_chunks",
    #    chunk_size=256,
    #    chunk_overlap=64
    #)
    
    # Example 2: Semantic chunking
    # process_directory(
    #     directory_path=json_directory,
    #     chunking_method="semantic",
    #     collection_name="news_semantic_chunks",
    #     model_name="mini"
    # )
    
    # Example 3: Document structure chunking
    # process_directory(
    #     directory_path=json_directory,
    #     chunking_method="document_structure",
    #     collection_name="news_structure_chunks"
    # )
    
    # Example 4: Hybrid chunking
    # process_directory(
    #     directory_path=json_directory,
    #     chunking_method="hybrid",
    #     collection_name="news_hybrid_chunks",
    #     chunk_size=256,
    #     chunk_overlap=64,
    #     semantic_model="mini"
    # )

### Test on a subset

In [45]:
import json
import os
import uuid
from pathlib import Path
from typing import List, Dict, Any
import chromadb
from chromadb.utils import embedding_functions

def get_multilingual_embedding_function(model_name: str = "multilingual-mpnet"):
    """Get ChromaDB-compatible multilingual embedding function."""
    
    multilingual_models = {
        "multilingual-mpnet": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        "multilingual-minilm": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", 
        "bge-m3": "BAAI/bge-m3",
    }
    
    model_path = multilingual_models.get(model_name, multilingual_models["multilingual-mpnet"])
    return embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name=model_path,
        device="cpu"
    )

def detect_language(text: str) -> str:
    """Simple language detection for EN/DE."""
    german_indicators = ['der', 'die', 'das', 'und', 'ist', 'für', 'von', 'mit', 'zu', 'auf']
    english_indicators = ['the', 'and', 'is', 'for', 'of', 'with', 'to', 'on', 'in', 'at']
    
    text_lower = text.lower()
    german_count = sum(1 for word in german_indicators if word in text_lower)
    english_count = sum(1 for word in english_indicators if word in text_lower)
    
    return "de" if german_count > english_count else "en"

def simple_chunk_text(text: str, chunk_size: int = 200) -> List[str]:
    """Simple text chunking for testing."""
    if not text or len(text.strip()) == 0:
        return []
    
    words = text.split()
    chunks = []
    current_chunk = []
    current_length = 0
    
    for word in words:
        if current_length + len(word) + 1 > chunk_size and current_chunk:
            chunks.append(' '.join(current_chunk))
            current_chunk = [word]
            current_length = len(word)
        else:
            current_chunk.append(word)
            current_length += len(word) + 1
    
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    
    return chunks

def process_test_files(json_directory: str, max_files: int = 5):
    """Process a small subset of JSON files for testing."""
    print(f"🧪 TESTING MULTILINGUAL CHUNKING")
    print("="*40)
    
    directory = Path(json_directory)
    json_files = list(directory.glob("*.json"))[:max_files]  # Take only first few files
    
    if not json_files:
        print(f"❌ No JSON files found in {json_directory}")
        return []
    
    print(f"📁 Testing with {len(json_files)} files:")
    for f in json_files:
        print(f"   • {f.name}")
    
    all_chunks = []
    language_counts = {"en": 0, "de": 0, "other": 0}
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            file_language = data.get('language', 'unknown')
            print(f"\n📄 Processing: {json_file.name} (language: {file_language})")
            
            # Get paragraphs
            paragraphs = data.get('paragraphs', [])
            file_chunks = 0
            
            for para_idx, paragraph in enumerate(paragraphs[:3]):  # Limit to first 3 paragraphs
                # Handle paragraph structure
                if isinstance(paragraph, dict):
                    heading = paragraph.get('heading', '')
                    text = paragraph.get('text', '')
                    full_text = f"{heading}\n{text}" if heading else text
                    paragraph_heading = heading
                else:
                    full_text = paragraph
                    paragraph_heading = ""
                
                if not full_text or len(full_text.strip()) == 0:
                    continue
                
                # Simple chunking
                chunks = simple_chunk_text(full_text, chunk_size=200)
                
                for chunk_idx, chunk_text in enumerate(chunks):
                    chunk_data = {
                        "id": str(uuid.uuid4()),
                        "text": chunk_text,
                        "metadata": {
                            "filename": data.get('filename', json_file.name),
                            "title": data.get('title', 'Unknown'),
                            "language": data.get('language', 'unknown'),  # Use existing language metadata
                            "date": data.get('date', ''),
                            "source_path": data.get('source_path', ''),
                            "paragraph_index": para_idx,
                            "paragraph_heading": paragraph_heading,
                            "chunk_index": chunk_idx,
                            "chunk_method": "simple_test",
                            "chunk_length": len(chunk_text),
                            # Add original metadata
                            "keywords": ", ".join(data.get('keywords', [])) if isinstance(data.get('keywords'), list) else str(data.get('keywords', '')),
                            # Keep detected language as backup, but prioritize existing language
                            "detected_language": detect_language(chunk_text),
                            "original_filename": data.get('original_filename', '')
                        }
                    }
                    all_chunks.append(chunk_data)
                    file_chunks += 1
                    
                    # Count languages
                    lang = data.get('language', 'unknown')
                    if lang == 'en':
                        language_counts['en'] += 1
                    elif lang == 'de':
                        language_counts['de'] += 1
                    else:
                        language_counts['other'] += 1
            
            print(f"   Generated {file_chunks} chunks")
            
        except Exception as e:
            print(f"   ❌ Error processing {json_file}: {e}")
    
    print(f"\n📊 Total test chunks: {len(all_chunks)}")
    print(f"🌍 Language distribution:")
    for lang, count in language_counts.items():
        if count > 0:
            print(f"   {lang}: {count} chunks")
    
    return all_chunks

def save_test_collection(chunks_data: List[Dict[str, Any]], collection_name: str = "test_multilingual"):
    """Save test chunks to ChromaDB."""
    print(f"\n💾 SAVING TO CHROMADB")
    print("="*25)
    
    try:
        # Initialize ChromaDB
        client = chromadb.PersistentClient(path="./chroma_db")
        
        # Delete existing test collection
        try:
            client.delete_collection(name=collection_name)
            print(f"🗑️  Deleted existing test collection")
        except:
            pass
        
        # Create collection with multilingual embeddings
        embedding_function = get_multilingual_embedding_function("multilingual-minilm")  # Faster for testing
        
        collection = client.create_collection(
            name=collection_name,
            embedding_function=embedding_function
        )
        print(f"✅ Created collection: {collection_name}")
        
        # Prepare data
        documents = [chunk["text"] for chunk in chunks_data]
        metadatas = [chunk["metadata"] for chunk in chunks_data]
        ids = [chunk["id"] for chunk in chunks_data]
        
        print(f"🔄 Generating embeddings for {len(documents)} chunks...")
        
        # Add in one batch (small test set)
        collection.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
        
        final_count = collection.count()
        print(f"✅ Saved {final_count} chunks successfully")
        
        return True
        
    except Exception as e:
        print(f"❌ Error saving to ChromaDB: {e}")
        return False

def test_multilingual_search(collection_name: str = "test_multilingual"):
    """Test multilingual search functionality."""
    print(f"\n🔍 TESTING MULTILINGUAL SEARCH")
    print("="*35)
    
    try:
        client = chromadb.PersistentClient(path="./chroma_db")
        collection = client.get_collection(name=collection_name)
        
        # Check language distribution
        sample = collection.get(limit=100, include=['metadatas'])
        languages = [meta.get('language', 'unknown') for meta in sample['metadatas']]
        lang_counts = {}
        for lang in languages:
            lang_counts[lang] = lang_counts.get(lang, 0) + 1
        
        print(f"📊 Language distribution in sample:")
        for lang, count in lang_counts.items():
            print(f"   {lang}: {count} chunks")
        
        # Test queries
        test_queries = [
            ("ETH president", None),  # No language filter
            ("student research", "en"),  # English only
            ("university", "de"),  # German only (if available)
            ("research projects", None),
        ]
        
        for query, lang_filter in test_queries:
            filter_text = f" (filter: {lang_filter})" if lang_filter else ""
            print(f"\n🔍 Query: '{query}'{filter_text}")
            
            # Build where clause for language filtering
            where_clause = {"language": lang_filter} if lang_filter else None
            
            results = collection.query(
                query_texts=[query],
                n_results=3,
                where=where_clause,
                include=['documents', 'metadatas', 'distances']
            )
            
            if results['documents'][0]:
                for i, (doc, meta, distance) in enumerate(zip(
                    results['documents'][0],
                    results['metadatas'][0], 
                    results['distances'][0]
                )):
                    similarity = 1 / (1 + distance)
                    filename = meta.get('filename', 'unknown')
                    doc_lang = meta.get('language', 'unknown')
                    title = meta.get('title', 'No title')
                    
                    print(f"   {i+1}. {title} ({doc_lang}) - similarity: {similarity:.3f}")
                    print(f"      File: {filename}")
                    print(f"      Text: {doc[:80]}...")
            else:
                print("   No results found")
        
        return True
        
    except Exception as e:
        print(f"❌ Search test failed: {e}")
        return False

def run_full_test(json_directory: str):
    """Run the complete test suite."""
    print("🚀 MULTILINGUAL CHUNKING TEST SUITE")
    print("="*50)
    
    # Step 1: Process test files
    chunks = process_test_files(json_directory, max_files=5)
    
    if not chunks:
        print("❌ No chunks generated, test failed")
        return False
    
    # Step 2: Save to ChromaDB
    success = save_test_collection(chunks, "test_multilingual")
    
    if not success:
        print("❌ Failed to save to ChromaDB")
        return False
    
    # Step 3: Test search
    search_success = test_multilingual_search("test_multilingual")
    
    if search_success:
        print(f"\n🎉 TEST COMPLETED SUCCESSFULLY!")
        print(f"   • Processed {len(chunks)} chunks")
        print(f"   • Multilingual embeddings working")
        print(f"   • Search functionality verified")
        print(f"\n✅ Ready to process full dataset!")
        return True
    else:
        print(f"\n❌ Test failed at search step")
        return False

# Usage
if __name__ == "__main__":
    # Update this path to your JSON files
    json_directory = "/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2"
    
    # Run the test
    run_full_test(json_directory)

🚀 MULTILINGUAL CHUNKING TEST SUITE
🧪 TESTING MULTILINGUAL CHUNKING
📁 Testing with 5 files:
   • sunbathing-meteoroids.json
   • de_news_events_2016_02_radical-cairo-in-shenzhen.json
   • die-eth-zuerich-nimmt-kurs-auf-netto-null.json
   • how-zurich-has-to-change-its-roads-to-have-more-e-bikes-than-cars.json
   • wer-in-singapur-forschen-will-soll-sich-jetzt-melden.json

📄 Processing: sunbathing-meteoroids.json (language: en)
   Generated 20 chunks

📄 Processing: de_news_events_2016_02_radical-cairo-in-shenzhen.json (language: en)
   Generated 15 chunks

📄 Processing: die-eth-zuerich-nimmt-kurs-auf-netto-null.json (language: de)
   Generated 9 chunks

📄 Processing: how-zurich-has-to-change-its-roads-to-have-more-e-bikes-than-cars.json (language: en)
   Generated 23 chunks

📄 Processing: wer-in-singapur-forschen-will-soll-sich-jetzt-melden.json (language: de)
   Generated 40 chunks

📊 Total test chunks: 107
🌍 Language distribution:
   en: 58 chunks
   de: 49 chunks

💾 SAVING TO CHROMADB

No sentence-transformers model found with name sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2. Creating a new one with mean pooling.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


✅ Created collection: test_multilingual
🔄 Generating embeddings for 107 chunks...
✅ Saved 107 chunks successfully

🔍 TESTING MULTILINGUAL SEARCH
📊 Language distribution in sample:
   en: 58 chunks
   de: 42 chunks

🔍 Query: 'ETH president'
   1. Wer In Singapur Forschen Will Soll Sich Jetzt Melden (de) - similarity: 0.044
      File: wer-in-singapur-forschen-will-soll-sich-jetzt-melden.md
      Text: würde auch erlauben, gemeinsame Finanzierungsprogramme zu entwickeln. Manu Kapur...
   2. Wer In Singapur Forschen Will Soll Sich Jetzt Melden (de) - similarity: 0.044
      File: wer-in-singapur-forschen-will-soll-sich-jetzt-melden.md
      Text: (in Englisch) auf der Seite des Singapore-ETH Centre ....
   3. Wer In Singapur Forschen Will Soll Sich Jetzt Melden (de) - similarity: 0.043
      File: wer-in-singapur-forschen-will-soll-sich-jetzt-melden.md
      Text: zu schlagen. Was macht der Direktor des ETH-Zentrums Singapur (SEC) genau? Der D...

🔍 Query: 'student research' (filter: en)


### Run for all files

In [46]:
#Process with Multilingual Embeddings:
json_directory = "/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2"

# Best for EN/DE performance
process_directory(
    directory_path=json_directory,
    chunking_method="recursive",
    collection_name="news_multilingual_recursive", 
    embedding_model="multilingual-mpnet",
    chunk_size=256,
    chunk_overlap=64
)

# Excellent alternative
process_directory(
    directory_path=json_directory,
    chunking_method="document_structure",
    collection_name="news_bge_m3_chunks",
    embedding_model="bge-m3"
)

Found 4398 JSON files
Using chunking method: recursive
Processing: sunbathing-meteoroids.json
  Generated 24 chunks
Processing: de_news_events_2016_02_radical-cairo-in-shenzhen.json
  Generated 22 chunks
Processing: die-eth-zuerich-nimmt-kurs-auf-netto-null.json
  Generated 29 chunks
Processing: how-zurich-has-to-change-its-roads-to-have-more-e-bikes-than-cars.json
  Generated 49 chunks
Processing: wer-in-singapur-forschen-will-soll-sich-jetzt-melden.json
  Generated 44 chunks
Processing: recovering-hidden-treasures-and-building-boats.json
  Generated 28 chunks
Processing: responsive-ergebnisanzeige-im-wissensportal.json
  Generated 1 chunks
Processing: die-lehren-des-fernunterrichts-in-der-mathematik.json
  Generated 24 chunks
Processing: de_news_events_2019_07_herzklappen-aus-silikon.json
  Generated 35 chunks
Processing: erste-erkenntnisse-aus-der-befragung-zu-future-of-work.json
  Generated 42 chunks
Processing: der-inspirierende-blick-aus-dem-auto.json
  Generated 40 chunks
Proces

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Created new collection: news_multilingual_recursive with multilingual-mpnet embeddings
Generating multilingual embeddings for 109128 chunks...
   Processed 500/109128 chunks
   Processed 1000/109128 chunks
   Processed 1500/109128 chunks
   Processed 2000/109128 chunks
   Processed 2500/109128 chunks
   Processed 3000/109128 chunks
   Processed 3500/109128 chunks
   Processed 4000/109128 chunks
   Processed 4500/109128 chunks
   Processed 5000/109128 chunks
   Processed 5500/109128 chunks
   Processed 6000/109128 chunks
   Processed 6500/109128 chunks
   Processed 7000/109128 chunks
   Processed 7500/109128 chunks
   Processed 8000/109128 chunks
   Processed 8500/109128 chunks
   Processed 9000/109128 chunks
   Processed 9500/109128 chunks
   Processed 10000/109128 chunks
   Processed 10500/109128 chunks
   Processed 11000/109128 chunks
   Processed 11500/109128 chunks
   Processed 12000/109128 chunks
   Processed 12500/109128 chunks
   Processed 13000/109128 chunks
   Processed 13500/

### Multilingual search

Test multilingual search with both collections news_multilingual_recursive and news_bge_m3_chunks

In [53]:
import chromadb
from typing import List, Dict, Any, Optional

def search_multilingual(collection_name: str, query: str, n_results: int = 10, 
                       language_filter: str = None, **additional_filters) -> List[Dict[str, Any]]:
    """
    Perform multilingual search in ChromaDB collection.
    
    Args:
        collection_name: Name of the ChromaDB collection
        query: Search query in English or German
        n_results: Number of results to return
        language_filter: Filter by language ('en', 'de', or None for all)
        **additional_filters: Additional metadata filters
    
    Returns:
        List of search results with text, metadata, and similarity scores
    """
    try:
        # Connect to ChromaDB
        client = chromadb.PersistentClient(path="./chroma_db")
        collection = client.get_collection(name=collection_name)
        
        # Build where clause for filtering
        conditions = []
        
        # Add language filter
        if language_filter:
            conditions.append({"language": {"$eq": language_filter}})
        
        # Add additional filters with proper operators
        for key, value in additional_filters.items():
            if key == "date":
                # For date filtering, create possible date variations
                # If user searches "2020", match "2020-01", "2020-02", etc.
                if len(str(value)) == 4:  # Year only
                    year = str(value)
                    possible_dates = [f"{year}-{month:02d}" for month in range(1, 13)]
                    conditions.append({key: {"$in": possible_dates}})
                else:
                    # Exact date match
                    conditions.append({key: {"$eq": str(value)}})
            elif key in ["keywords", "entities", "ner_entities"]:
                # For text fields, we can't do partial matching with supported operators
                # So we'll get all results and filter in Python afterward
                # For now, skip these filters in the where clause
                continue
            else:
                # Exact match for other fields
                conditions.append({key: {"$eq": str(value)}})
        
        # Combine conditions with $and if multiple conditions exist
        where_clause = None
        if len(conditions) == 1:
            where_clause = conditions[0]
        elif len(conditions) > 1:
            where_clause = {"$and": conditions}
        
        print(f"🔍 Search query: '{query}'")
        if where_clause:
            print(f"🎛️  Filters applied: {where_clause}")
        
        # Perform search with higher n_results if we need to filter afterward
        search_limit = n_results * 3 if any(k in ["keywords", "entities", "ner_entities"] for k in additional_filters.keys()) else n_results
        
        results = collection.query(
            query_texts=[query],
            n_results=search_limit,
            where=where_clause,
            include=['documents', 'metadatas', 'distances']
        )
        
        # Post-process results for text-based filtering
        filtered_results = []
        for doc, meta, distance in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
            # Apply text-based filters
            include_result = True
            
            for key, value in additional_filters.items():
                if key == "keywords":
                    keywords_field = meta.get('keywords', '')
                    if value.lower() not in keywords_field.lower():
                        include_result = False
                        break
                elif key == "entities":
                    entities_field = meta.get('ner_entities', '')
                    if value.lower() not in entities_field.lower():
                        include_result = False
                        break
            
            if include_result:
                similarity = 1 / (1 + distance)
                
                filtered_results.append({
                    'text': doc,
                    'metadata': meta,
                    'similarity': similarity,
                    'distance': distance,
                    'title': meta.get('title', 'No title'),
                    'filename': meta.get('filename', 'Unknown'),
                    'language': meta.get('language', 'unknown'),
                    'date': meta.get('date', ''),
                    'source_path': meta.get('source_path', ''),
                    'keywords': meta.get('keywords', '')
                })
                
                if len(filtered_results) >= n_results:
                    break
        
        return filtered_results[:n_results]
        
    except Exception as e:
        print(f"❌ Error in multilingual search: {e}")
        return []

def display_search_results(results: List[Dict[str, Any]], max_text_length: int = 150):
    """Display search results in a formatted way."""
    if not results:
        print("No results found.")
        return
    
    print(f"📊 Found {len(results)} results:")
    print("="*60)
    
    for i, result in enumerate(results, 1):
        title = result['title']
        language = result['language']
        similarity = result['similarity']
        text = result['text']
        filename = result['filename']
        
        # Truncate text if too long
        display_text = text[:max_text_length] + "..." if len(text) > max_text_length else text
        
        print(f"\n{i}. {title} ({language})")
        print(f"   Similarity: {similarity:.3f} | File: {filename}")
        print(f"   Text: {display_text}")

def demo_multilingual_search(collection_name: str = "news_multilingual_recursive"):
    """Demo function showing different search capabilities."""
    
    print("🌍 MULTILINGUAL SEARCH DEMO")
    print("="*40)
    
    # Test queries
    test_cases = [
        {
            "description": "English query - no filter",
            "query": "ETH research projects",
            "language_filter": None,
            "n_results": 3
        },
        {
            "description": "German query - no filter", 
            "query": "ETH Forschungsprojekte",
            "language_filter": None,
            "n_results": 3
        },
        {
            "description": "English query - English only",
            "query": "research",
            "language_filter": "en",
            "n_results": 3
        },
        {
            "description": "German query - German only",
            "query": "Forschung",
            "language_filter": "de", 
            "n_results": 3
        },
        {
            "description": "Cross-language search - German query, English filter",
            "query": "Universität",
            "language_filter": "en",
            "n_results": 3
        }
    ]
    
    for test_case in test_cases:
        print(f"\n🔍 Test: {test_case['description']}")
        print(f"Query: '{test_case['query']}'")
        if test_case['language_filter']:
            print(f"Language filter: {test_case['language_filter']}")
        print("-" * 50)
        
        results = search_multilingual(
            collection_name=collection_name,
            query=test_case['query'],
            n_results=test_case['n_results'],
            language_filter=test_case['language_filter']
        )
        
        display_search_results(results, max_text_length=100)

def search_with_metadata_filters(collection_name: str, query: str, **filters):
    """Example of searching with additional metadata filters."""
    
    print(f"\n🎯 ADVANCED SEARCH WITH FILTERS")
    print(f"Query: '{query}'")
    print(f"Filters: {filters}")
    print("-" * 40)
    
    results = search_multilingual(
        collection_name=collection_name,
        query=query,
        n_results=5,
        **filters
    )
    
    if results:
        display_search_results(results)
        print(f"\n📊 Found {len(results)} results")
        
        # Show metadata from first result to understand data structure
        if results:
            print(f"\n🔍 Sample metadata from first result:")
            sample_meta = results[0]['metadata']
            for key, value in sample_meta.items():
                if value:  # Only show non-empty values
                    display_value = str(value)[:50] + "..." if len(str(value)) > 50 else str(value)
                    print(f"   {key}: {display_value}")
    else:
        print("No results found. Let's check what metadata is available...")
        
        # If no results, show available metadata structure
        try:
            client = chromadb.PersistentClient(path="./chroma_db")
            collection = client.get_collection(name=collection_name)
            sample = collection.get(limit=3, include=['metadatas'])
            
            if sample['metadatas']:
                print(f"\n📋 Available metadata fields in collection:")
                all_keys = set()
                for meta in sample['metadatas']:
                    all_keys.update(meta.keys())
                
                for key in sorted(all_keys):
                    sample_values = [meta.get(key, '') for meta in sample['metadatas'] if meta.get(key)]
                    if sample_values:
                        example = str(sample_values[0])[:30] + "..." if len(str(sample_values[0])) > 30 else str(sample_values[0])
                        print(f"   • {key}: (example: {example})")
        except Exception as e:
            print(f"Could not retrieve sample metadata: {e}")
    
    return results

# Usage examples with news_multilingual_recursive
if __name__ == "__main__":
    collection_name = "news_multilingual_recursive"  # Update with your collection name
    
    # Basic searches
    print("🌍 BASIC MULTILINGUAL SEARCHES")
    print("="*50)
    
    # English query
    print("\n1. English query:")
    results = search_multilingual(collection_name, "ETH research projects", n_results=5)
    display_search_results(results)
    
    # German query  
    print("\n2. German query:")
    results = search_multilingual(collection_name, "ETH Forschungsprojekte", n_results=5)
    display_search_results(results)
    
    # Filter by language
    print("\n3. English filter:")
    results = search_multilingual(collection_name, "research", language_filter="en", n_results=5)
    display_search_results(results)
    
    # Advanced filtering examples
    print("\n\n🎯 ADVANCED FILTERING EXAMPLES")
    print("="*50)
    
    # Filter by date range (year)
    search_with_metadata_filters(
        collection_name, 
        "student projects",
        language_filter="en",
        date="2020"  # Will match 2020-01, 2020-02, etc.
    )
    
    # Filter by exact date
    search_with_metadata_filters(
        collection_name,
        "research",
        date="2017-03"  # Exact date match
    )
    
    # Filter by keywords (post-processed)
    search_with_metadata_filters(
        collection_name,
        "artificial intelligence", 
        keywords="research"  # Must contain "research" in keywords field
    )
    
    # Test simple language filter (should work)
    search_with_metadata_filters(
        collection_name,
        "university",
        language_filter="en"
    )
    
    # Run full demo
    print("\n\n" + "="*60)
    demo_multilingual_search(collection_name)

🌍 BASIC MULTILINGUAL SEARCHES

1. English query:
🔍 Search query: 'ETH research projects'
📊 Found 5 results:

1. 50 Million Swiss Francs For Institute Of Theoretical Studies (en)
   Similarity: 0.294 | File: 50-million-swiss-francs-for-institute-of-theoretical-studies.md
   Text: Contribution to ETH and Beyond

2. Die Eth Gastronomie In Corona Zeiten (de)
   Similarity: 0.281 | File: die-eth-gastronomie-in-corona-zeiten.md
   Text: Enge Zusammenarbeit mit der ETH

3. Ruecktritt Nach Acht Jahren Einsatz Fuer Die Forschung (de)
   Similarity: 0.281 | File: ruecktritt-nach-acht-jahren-einsatz-fuer-die-forschung.md
   Text: Leidenschaft für die Forschung an der ETH

4. Ruecktritt Nach Acht Jahren Einsatz Fuer Die Forschung (de)
   Similarity: 0.281 | File: ruecktritt-nach-acht-jahren-einsatz-fuer-die-forschung.md
   Text: Leidenschaft für die Forschung an der ETH

5. 50 Millionen Spende Fuer Institut Fuer Theoretische Studien (de)
   Similarity: 0.280 | File: 50-millionen-spende-fuer-instit

In [54]:
# Usage examples for the collection news_bge_m3_chunks
if __name__ == "__main__":
    collection_name = "news_bge_m3_chunks"  # Update with your collection name
    
    # Basic searches
    print("🌍 BASIC MULTILINGUAL SEARCHES")
    print("="*50)
    
    # English query
    print("\n1. English query:")
    results = search_multilingual(collection_name, "ETH research projects", n_results=5)
    display_search_results(results)
    
    # German query  
    print("\n2. German query:")
    results = search_multilingual(collection_name, "ETH Forschungsprojekte", n_results=5)
    display_search_results(results)
    
    # Filter by language
    print("\n3. English filter:")
    results = search_multilingual(collection_name, "research", language_filter="en", n_results=5)
    display_search_results(results)
    
    # Advanced filtering examples
    print("\n\n🎯 ADVANCED FILTERING EXAMPLES")
    print("="*50)
    
    # Filter by date range (year)
    search_with_metadata_filters(
        collection_name, 
        "student projects",
        language_filter="en",
        date="2020"  # Will match 2020-01, 2020-02, etc.
    )
    
    # Filter by exact date
    search_with_metadata_filters(
        collection_name,
        "research",
        date="2017-03"  # Exact date match
    )
    
    # Filter by keywords (post-processed)
    search_with_metadata_filters(
        collection_name,
        "artificial intelligence", 
        keywords="research"  # Must contain "research" in keywords field
    )
    
    # Test simple language filter (should work)
    search_with_metadata_filters(
        collection_name,
        "university",
        language_filter="en"
    )
    
    # Run full demo
    print("\n\n" + "="*60)
    demo_multilingual_search(collection_name)

🌍 BASIC MULTILINGUAL SEARCHES

1. English query:
🔍 Search query: 'ETH research projects'
📊 Found 5 results:

1. Die Eth Bibliothek Unterstuetzt Die Open Library Of Humanities (de)
   Similarity: 0.228 | File: de_internal_2022_06_die-eth-bibliothek-unterstuetzt-die-open-library-of-humanities.md
   Text: Weitere News der ETH-​Bibliothek

2. Schnuppersemester Fuer Fluechtlinge (de)
   Similarity: 0.215 | File: schnuppersemester-fuer-fluechtlinge.md
   Text: Organisiert wird das Projekt vom Verband der ETH-Studierenden VSETH mit Unterstützung der ETH-Schulleitung .

3. Informationsveranstaltung (de)
   Similarity: 0.204 | File: informationsveranstaltung.md
   Text: Auch gibt es im ETH-Bereich (Also die beiden ETHs und die Forschungsinstitute des ETH-Bereichs) das Projekt, «Fix the leaky Pipeline», das junge Forsc...

4. Mehr Energieeffizienz Und Intensivierte Energieforschung Im Eth Bereich (de)
   Similarity: 0.200 | File: de_news_events_2014_11_mehr-energieeffizienz-und-intensivierte-ene

### Check existing collections

In [49]:
import chromadb
from pathlib import Path

def check_chromadb_counts(db_path: str = "./chroma_db"):
    """Simple check of ChromaDB chunk counts."""
    print("🔍 CHECKING CHROMADB CHUNKS")
    print("="*30)
    
    try:
        # Connect to ChromaDB
        client = chromadb.PersistentClient(path=db_path)
        
        # List all collections
        collections = client.list_collections()
        
        if not collections:
            print("❌ No collections found")
            return
        
        print(f"📊 Found {len(collections)} collection(s):")
        print()
        
        total_chunks = 0
        
        for collection in collections:
            count = collection.count()
            total_chunks += count
            
            print(f"📁 {collection.name}")
            print(f"   Chunks: {count:,}")
            
            # Get one sample chunk to verify data exists
            if count > 0:
                try:
                    sample = collection.get(limit=1, include=['documents', 'metadatas'])
                    if sample['documents']:
                        doc_length = len(sample['documents'][0])
                        filename = sample['metadatas'][0].get('filename', 'unknown')
                        print(f"   Sample: {doc_length} chars from '{filename}'")
                except:
                    print(f"   ⚠️  Error reading sample data")
            print()
        
        print(f"📈 TOTAL CHUNKS ACROSS ALL COLLECTIONS: {total_chunks:,}")
        
        # Quick success check
        if total_chunks > 50000:
            print("✅ Looks good! Large number of chunks found")
        elif total_chunks > 1000:
            print("🟡 Moderate number of chunks - might be partial")
        else:
            print("❌ Low chunk count - possible issue")
            
    except Exception as e:
        print(f"❌ Error checking ChromaDB: {e}")

def check_specific_collection(collection_name: str, db_path: str = "./chroma_db"):
    """Check a specific collection."""
    print(f"🔍 CHECKING COLLECTION: {collection_name}")
    print("="*40)
    
    try:
        client = chromadb.PersistentClient(path=db_path)
        collection = client.get_collection(name=collection_name)
        
        count = collection.count()
        print(f"📊 Total chunks: {count:,}")
        
        if count > 0:
            # Get sample
            sample = collection.get(limit=3, include=['documents', 'metadatas'])
            
            print(f"\n📝 Sample chunks:")
            for i, (doc, meta) in enumerate(zip(sample['documents'], sample['metadatas'])):
                filename = meta.get('filename', 'unknown')
                method = meta.get('chunk_method', 'unknown')
                print(f"   {i+1}. {filename} ({method}) - {len(doc)} chars")
                print(f"      {doc[:60]}{'...' if len(doc) > 60 else ''}")
        
        return count
        
    except Exception as e:
        print(f"❌ Collection '{collection_name}' not found or error: {e}")
        return 0

def quick_status():
    """Super quick status check."""
    try:
        client = chromadb.PersistentClient(path="./chroma_db")
        collections = client.list_collections()
        
        print("📊 QUICK STATUS:")
        for col in collections:
            count = col.count()
            print(f"   {col.name}: {count:,} chunks")
            
    except Exception as e:
        print(f"❌ Error: {e}")

if __name__ == "__main__":
    # Quick check of all collections
    check_chromadb_counts()
    
    # Or check specific collection:
    #check_specific_collection("news_multilingual_recursive") #news_multilingual_recursive, news_bge_m3_chunks
    
    # Or super quick status:
    # quick_status()

🔍 CHECKING CHROMADB CHUNKS
📊 Found 4 collection(s):

📁 news_bge_m3_chunks
   Chunks: 56,977
   Sample: 480 chars from 'sunbathing-meteoroids.md'

📁 news_structure_chunks
   Chunks: 56,977
   Sample: 480 chars from 'sunbathing-meteoroids.md'

📁 news_multilingual_recursive
   Chunks: 109,128
   Sample: 32 chars from 'sunbathing-meteoroids.md'

📁 news_recursive_chunks
   Chunks: 109,128
   Sample: 32 chars from 'sunbathing-meteoroids.md'

📈 TOTAL CHUNKS ACROSS ALL COLLECTIONS: 332,210
✅ Looks good! Large number of chunks found


# dense retrieval

In [26]:
%pip install openai

Python(62063) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.4/720.4 kB 4.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
import chromadb
import json
from openai import OpenAI
import numpy as np
from typing import List, Dict, Any
import statistics

# Initialize OpenAI client (set your API key)
openai_client = OpenAI(api_key="sk-proj-b3II8mJlx37d8xC-7eWsG5ex4THRWN7nPMPVTcTYuFdzH2d2Y54usmJOnVKEXIj33zw10E5JutT3BlbkFJWy54QoF50cMgLRZqZUCg9fJ0-jaOp-tW9NT1UFHx8rTcEm2tMIa0Y3pnMVSTP5yAotB3V3wiA")  # Replace with your key

def load_benchmark_questions(file_path: str = "eth_rag_dataset.json") -> List[Dict]:
    """Load benchmark questions and answers."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        # Fallback to hardcoded questions
        return [
            {"question": "Who was president of ETH in 2003?", "answer": "Olaf Kübler"},
            {"question": "Who were the rectors of ETH between 2017 and 2022?", "answer": "Sarah Springman, Günther Dissertori"},
            {"question": "Who at ETH received ERC grants?", "answer": "Multiple researchers"}
        ]

def search_with_chunks_final(collection, query: str, chunk_method: str, k: int = 10):
    """Search and filter by chunk method."""
    # Get more results than needed since we'll filter
    results = collection.query(
        query_texts=[query],
        n_results=k * 3,  # Get extra to account for filtering
        include=['documents', 'metadatas', 'distances']
    )
    
    # Filter by chunk method and compute similarity
    filtered_results = []
    for doc, meta, distance in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        if meta.get('chunk_method') == chunk_method:
            similarity = 1 / (1 + distance)  # Convert distance to similarity (0-1)
            filtered_results.append({
                'document': doc,
                'metadata': meta,
                'similarity': similarity,
                'distance': distance
            })
        
        if len(filtered_results) >= k:
            break
    
    return filtered_results[:k]

def evaluate_relevance_gpt(question: str, answer: str, documents: List[str]) -> List[float]:
    """Use GPT-3.5 to score document relevance."""
    prompt = f"""
Question: {question}
Expected Answer: {answer}

Rate each document's relevance to answering the question:
- 1.0: Highly relevant, contains the answer
- 0.5: Partially relevant, contains related information
- 0.0: Not relevant

Documents:
{chr(10).join([f"{i+1}. {doc[:200]}..." for i, doc in enumerate(documents)])}

Return only comma-separated scores (e.g., "1.0,0.5,0.0"):
"""
    
    try:
        response = openai_client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50,
            temperature=0
        )
        
        scores_text = response.choices[0].message.content.strip()
        scores = [float(x.strip()) for x in scores_text.split(',')]
        return scores[:len(documents)]  # Ensure we don't have extra scores
        
    except Exception as e:
        print(f"GPT evaluation failed: {e}")
        return [0.0] * len(documents)  # Fallback to zero scores

def compute_metrics(relevance_scores: List[float], k: int = 5):
    """Compute IR metrics."""
    if not relevance_scores:
        return {"precision": 0, "recall": 0, "mrr": 0, "ndcg": 0}
    
    # Precision@k
    relevant = sum(1 for score in relevance_scores[:k] if score > 0.5)
    precision = relevant / min(k, len(relevance_scores))
    
    # Recall@k (assumes all relevant docs are in the collection)
    total_relevant = sum(1 for score in relevance_scores if score > 0.5)
    recall = relevant / max(total_relevant, 1)
    
    # MRR (Mean Reciprocal Rank)
    mrr = 0
    for i, score in enumerate(relevance_scores):
        if score > 0.5:
            mrr = 1 / (i + 1)
            break
    
    # NDCG@k
    def dcg(scores):
        return sum(score / np.log2(i + 2) for i, score in enumerate(scores))
    
    actual_dcg = dcg(relevance_scores[:k])
    ideal_scores = sorted(relevance_scores, reverse=True)[:k]
    ideal_dcg = dcg(ideal_scores)
    ndcg = actual_dcg / ideal_dcg if ideal_dcg > 0 else 0
    
    return {
        "precision": precision,
        "recall": recall, 
        "mrr": mrr,
        "ndcg": ndcg
    }

def evaluate_chunking_methods(db_path: str = "./chroma_db", k: int = 5):
    """Main evaluation function."""
    print("🔍 CHUNKING METHODS EVALUATION")
    print("="*40)
    
    # Connect to ChromaDB
    client = chromadb.PersistentClient(path=db_path)
    
    collections = {
        "structure": client.get_collection("news_structure_chunks"),
        "recursive": client.get_collection("news_recursive_chunks")
    }
    
    # Load benchmark questions
    benchmark_data = load_benchmark_questions()
    print(f"📊 Loaded {len(benchmark_data)} benchmark questions")
    
    # Results storage
    results = {"structure": [], "recursive": []}
    
    # Evaluate each question
    for i, item in enumerate(benchmark_data[:5]):  # Limit to 5 questions to minimize cost
        question = item["question"]
        expected_answer = item["answer"]
        
        print(f"\n🔍 Question {i+1}: {question}")
        
        for method_name, collection in collections.items():
            chunk_method = "document_structure" if method_name == "structure" else "recursive"
            
            # Search
            search_results = search_with_chunks_final(collection, question, chunk_method, k)
            
            if search_results:
                documents = [r['document'] for r in search_results]
                
                # Evaluate with GPT
                relevance_scores = evaluate_relevance_gpt(question, expected_answer, documents)
                
                # Compute metrics
                metrics = compute_metrics(relevance_scores, k)
                
                results[method_name].append({
                    "question": question,
                    "metrics": metrics,
                    "scores": relevance_scores
                })
                
                print(f"   {method_name}: P@{k}={metrics['precision']:.3f}, MRR={metrics['mrr']:.3f}")
            else:
                print(f"   {method_name}: No results found")
    
    # Aggregate results
    print(f"\n📊 FINAL RESULTS")
    print("="*30)
    
    for method_name, method_results in results.items():
        if method_results:
            avg_precision = statistics.mean([r['metrics']['precision'] for r in method_results])
            avg_recall = statistics.mean([r['metrics']['recall'] for r in method_results])
            avg_mrr = statistics.mean([r['metrics']['mrr'] for r in method_results])
            avg_ndcg = statistics.mean([r['metrics']['ndcg'] for r in method_results])
            
            print(f"\n🔧 {method_name.upper()} CHUNKING:")
            print(f"   Precision@{k}: {avg_precision:.3f}")
            print(f"   Recall@{k}: {avg_recall:.3f}")
            print(f"   MRR: {avg_mrr:.3f}")
            print(f"   NDCG@{k}: {avg_ndcg:.3f}")
    
    return results

def quick_comparison():
    """Quick comparison without GPT evaluation."""
    print("🚀 QUICK COMPARISON (No GPT)")
    print("="*30)
    
    client = chromadb.PersistentClient(path="./chroma_db")
    collections = {
        "structure": client.get_collection("news_structure_chunks"),
        "recursive": client.get_collection("news_recursive_chunks")
    }
    
    test_queries = [
        "Who was president of ETH?",
        "ERC grants at ETH",
        "student research projects"
    ]
    
    for query in test_queries:
        print(f"\nQuery: '{query}'")
        
        for method_name, collection in collections.items():
            chunk_method = "document_structure" if method_name == "structure" else "recursive"
            results = search_with_chunks_final(collection, query, chunk_method, 3)
            
            avg_similarity = statistics.mean([r['similarity'] for r in results]) if results else 0
            print(f"   {method_name}: {len(results)} results, avg similarity: {avg_similarity:.3f}")

if __name__ == "__main__":
    # Set your OpenAI API key here
    openai_client = OpenAI(api_key="sk-proj-b3II8mJlx37d8xC-7eWsG5ex4THRWN7nPMPVTcTYuFdzH2d2Y54usmJOnVKEXIj33zw10E5JutT3BlbkFJWy54QoF50cMgLRZqZUCg9fJ0-jaOp-tW9NT1UFHx8rTcEm2tMIa0Y3pnMVSTP5yAotB3V3wiA")
    
    # Quick comparison (free)
    quick_comparison()
    
    # Full evaluation with GPT (costs money)
    evaluate_chunking_methods(k=5)

🚀 QUICK COMPARISON (No GPT)

Query: 'Who was president of ETH?'
   structure: 3 results, avg similarity: 0.539
   recursive: 3 results, avg similarity: 0.655

Query: 'ERC grants at ETH'
   structure: 3 results, avg similarity: 0.658
   recursive: 3 results, avg similarity: 0.750

Query: 'student research projects'
   structure: 3 results, avg similarity: 0.576
   recursive: 3 results, avg similarity: 0.603
🔍 CHUNKING METHODS EVALUATION
📊 Loaded 25 benchmark questions

🔍 Question 1: Who was president of ETH in 2003?
   structure: P@5=0.000, MRR=0.000
   recursive: P@5=0.000, MRR=0.000

🔍 Question 2: Who were the rectors of ETH between 2017 and 2022?
   structure: P@5=0.000, MRR=0.000
   recursive: P@5=0.200, MRR=0.333

🔍 Question 3: Who at ETH received ERC grants?
   structure: P@5=0.400, MRR=1.000
   recursive: P@5=0.600, MRR=1.000

🔍 Question 4: When did the InSight get to Mars?
   structure: P@5=0.200, MRR=1.000
   recursive: P@5=0.333, MRR=1.000

🔍 Question 5: What did Prof. Schuber

# Archive

## chunking methods

**This version does not save metadata in chromadb and is therefore disgarded.**
(Ideas from Cleantecg RAG Github repo): 

Features:
✅ Independent paragraph processing - Each paragraph chunked separately
✅ 4 chunking methods: recursive, semantic, document structure, hybrid
✅ ChromaDB integration - Saves chunks with rich metadata
✅ Batch processing - Handles entire directories
✅ Flexible configuration - Easy to switch between methods


Chunking Methods:

Recursive (your provided function) - Character-based with smart separators
Semantic - Uses embedding models to find semantic boundaries
Document Structure - Sentence-based chunks (2-3 sentences each)
Hybrid - Semantic first, recursive fallback for large chunks

In [ ]:
#%pip install langchain langchain_experimental

In [ ]:
%pip install langchain_huggingface

In [9]:
import json
import os
import uuid
from pathlib import Path
from typing import List, Dict, Any
import chromadb
from chromadb.config import Settings

# Langchain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

def get_recursive_splitter(chunk_size: int, chunk_overlap: int):
    """Get recursive character text splitter."""
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
        length_function=len,
    )

def get_embedding_models():
    """Initialize embedding models for semantic chunking."""
    model_kwargs = {"device": "cpu"}  # Change to "cuda" if GPU available
    
    embedding_models = {
        "mini": HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2", 
            model_kwargs=model_kwargs
        ),
        "bge-m3": HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3", 
            model_kwargs=model_kwargs
        ),
        "gte": HuggingFaceEmbeddings(
            model_name="Alibaba-NLP/gte-base-en-v1.5", 
            model_kwargs={**model_kwargs, "trust_remote_code": True}
        ),
    }
    return embedding_models

def chunk_paragraph_recursive(paragraph: str, chunk_size: int = 256, chunk_overlap: int = 64) -> List[str]:
    """Chunk a single paragraph using recursive text splitter."""
    if not paragraph or len(paragraph.strip()) == 0:
        return []
    
    splitter = get_recursive_splitter(chunk_size, chunk_overlap)
    chunks = splitter.split_text(paragraph)
    return [chunk.strip() for chunk in chunks if chunk.strip()]

def chunk_paragraph_semantic(paragraph: str, model_name: str = "mini") -> List[str]:
    """Chunk a single paragraph using semantic chunking."""
    if not paragraph or len(paragraph.strip()) == 0:
        return []
    
    embedding_models = get_embedding_models()
    if model_name not in embedding_models:
        print(f"Model {model_name} not found, using 'mini' instead")
        model_name = "mini"
    
    semantic_chunker = SemanticChunker(
        embeddings=embedding_models[model_name],
        breakpoint_threshold_type="percentile"
    )
    
    try:
        chunks = semantic_chunker.split_text(paragraph)
        return [chunk.strip() for chunk in chunks if chunk.strip()]
    except Exception as e:
        print(f"Semantic chunking failed, falling back to recursive: {e}")
        return chunk_paragraph_recursive(paragraph)

def chunk_paragraph_document_structure(paragraph: str) -> List[str]:
    """Chunk based on document structure (sentences, periods)."""
    if not paragraph or len(paragraph.strip()) == 0:
        return []
    
    # Split by sentences (periods followed by space and capital letter)
    import re
    sentences = re.split(r'(?<=\.)\s+(?=[A-Z])', paragraph)
    
    # Group sentences into chunks (max 2-3 sentences per chunk)
    chunks = []
    current_chunk = ""
    sentence_count = 0
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
            
        if sentence_count == 0:
            current_chunk = sentence
        else:
            current_chunk += " " + sentence
        
        sentence_count += 1
        
        # Create chunk after 2-3 sentences or if getting too long
        if sentence_count >= 2 and (sentence_count >= 3 or len(current_chunk) > 200):
            chunks.append(current_chunk.strip())
            current_chunk = ""
            sentence_count = 0
    
    # Add remaining text
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    
    return chunks

def chunk_paragraph_hybrid(paragraph: str, chunk_size: int = 256, chunk_overlap: int = 64, 
                          semantic_model: str = "mini") -> List[str]:
    """Hybrid approach: try semantic first, fallback to recursive if needed."""
    if not paragraph or len(paragraph.strip()) == 0:
        return []
    
    # Try semantic chunking first
    try:
        semantic_chunks = chunk_paragraph_semantic(paragraph, semantic_model)
        
        # If semantic chunks are too large, apply recursive chunking to each
        final_chunks = []
        for chunk in semantic_chunks:
            if len(chunk) > chunk_size * 1.5:  # If chunk is too large
                sub_chunks = chunk_paragraph_recursive(chunk, chunk_size, chunk_overlap)
                final_chunks.extend(sub_chunks)
            else:
                final_chunks.append(chunk)
        
        return final_chunks
    
    except Exception as e:
        print(f"Hybrid chunking failed, using recursive: {e}")
        return chunk_paragraph_recursive(paragraph, chunk_size, chunk_overlap)

def process_json_file(file_path: str, chunking_method: str = "recursive", **kwargs) -> List[Dict[str, Any]]:
    """Process a single JSON file and return chunks with metadata."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        chunks_data = []
        paragraphs = data.get('paragraphs', [])
        
        for para_idx, paragraph in enumerate(paragraphs):
            # Handle new structure where paragraphs are dictionaries
            if isinstance(paragraph, dict):
                heading = paragraph.get('heading', '')
                text = paragraph.get('text', '')
                
                # Combine heading and text for chunking
                full_text = f"{heading}\n{text}" if heading else text
                paragraph_heading = heading
            else:
                # Handle old structure where paragraphs are strings
                full_text = paragraph
                paragraph_heading = ""
            
            if not full_text or len(full_text.strip()) == 0:
                continue
            
            # Apply selected chunking method
            if chunking_method == "recursive":
                chunks = chunk_paragraph_recursive(
                    full_text, 
                    kwargs.get('chunk_size', 256), 
                    kwargs.get('chunk_overlap', 64)
                )
            elif chunking_method == "semantic":
                chunks = chunk_paragraph_semantic(
                    full_text, 
                    kwargs.get('model_name', 'mini')
                )
            elif chunking_method == "document_structure":
                chunks = chunk_paragraph_document_structure(full_text)
            elif chunking_method == "hybrid":
                chunks = chunk_paragraph_hybrid(
                    full_text,
                    kwargs.get('chunk_size', 256),
                    kwargs.get('chunk_overlap', 64),
                    kwargs.get('semantic_model', 'mini')
                )
            else:
                print(f"Unknown chunking method: {chunking_method}, using recursive")
                chunks = chunk_paragraph_recursive(full_text)
            
            # Create chunk metadata
            for chunk_idx, chunk_text in enumerate(chunks):
                chunk_data = {
                    "id": str(uuid.uuid4()),
                    "text": chunk_text,
                    "metadata": {
                        "filename": data.get('filename', ''),
                        "title": data.get('title', ''),
                        "language": data.get('language', ''),
                        "date": data.get('date', ''),
                        "source_path": data.get('source_path', ''),
                        "paragraph_index": para_idx,
                        "paragraph_heading": paragraph_heading,
                        "chunk_index": chunk_idx,
                        "chunk_method": chunking_method,
                        "original_paragraph_length": len(full_text),
                        "chunk_length": len(chunk_text)
                    }
                }
                chunks_data.append(chunk_data)
        
        return chunks_data
    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return []

def save_to_chromadb(chunks_data: List[Dict[str, Any]], collection_name: str = "news_chunks"):
    """Save chunks to ChromaDB."""
    try:
        # Initialize ChromaDB client
        client = chromadb.PersistentClient(path="./chroma_db")
        
        # Create or get collection
        try:
            collection = client.get_collection(name=collection_name)
            print(f"Using existing collection: {collection_name}")
        except:
            collection = client.create_collection(name=collection_name)
            print(f"Created new collection: {collection_name}")
        
        # Prepare data for ChromaDB
        documents = [chunk["text"] for chunk in chunks_data]
        metadatas = [chunk["metadata"] for chunk in chunks_data]
        ids = [chunk["id"] for chunk in chunks_data]
        
        # Add to collection in batches
        batch_size = 100
        for i in range(0, len(documents), batch_size):
            batch_docs = documents[i:i+batch_size]
            batch_metas = metadatas[i:i+batch_size]
            batch_ids = ids[i:i+batch_size]
            
            collection.add(
                documents=batch_docs,
                metadatas=batch_metas,
                ids=batch_ids
            )
        
        print(f"Successfully saved {len(chunks_data)} chunks to ChromaDB collection '{collection_name}'")
        return True
    
    except Exception as e:
        print(f"Error saving to ChromaDB: {e}")
        return False

def process_directory(directory_path: str, chunking_method: str = "recursive", 
                     collection_name: str = "news_chunks", **kwargs):
    """Process all JSON files in a directory and save to ChromaDB."""
    directory = Path(directory_path)
    json_files = list(directory.rglob("*.json"))
    
    if not json_files:
        print(f"No JSON files found in {directory_path}")
        return
    
    print(f"Found {len(json_files)} JSON files")
    print(f"Using chunking method: {chunking_method}")
    
    all_chunks = []
    
    for json_file in json_files:
        print(f"Processing: {json_file.name}")
        chunks = process_json_file(str(json_file), chunking_method, **kwargs)
        all_chunks.extend(chunks)
        print(f"  Generated {len(chunks)} chunks")
    
    print(f"\nTotal chunks generated: {len(all_chunks)}")
    
    # Save to ChromaDB
    if all_chunks:
        save_to_chromadb(all_chunks, collection_name)
    else:
        print("No chunks to save")


# Usage examples
#if __name__ == "__main__":
#    json_directory = "/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2"
    
    # Example 1: Recursive chunking
    # process_directory(
     #   directory_path=json_directory,
      #  chunking_method="recursive",
       # collection_name="news_recursive_chunks",
        #chunk_size=256,
        #chunk_overlap=64
    #)
    
    # Example 2: Semantic chunking
    # process_directory(
    #     directory_path=json_directory,
    #     chunking_method="semantic",
    #     collection_name="news_semantic_chunks",
    #     model_name="mini"
    # )
    
    # Example 3: Document structure chunking
    # process_directory(
    #     directory_path=json_directory,
    #     chunking_method="document_structure",
    #     collection_name="news_structure_chunks"
    # )
    
    # Example 4: Hybrid chunking
    # process_directory(
    #     directory_path=json_directory,
    #     chunking_method="hybrid",
    #     collection_name="news_hybrid_chunks",
    #     chunk_size=256,
    #     chunk_overlap=64,
    #     semantic_model="mini"
    # )

In [2]:
# Update the path
json_directory = "/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2"

# Try different methods:
process_directory(json_directory, "recursive", "news_recursive_chunks", chunk_size=256, chunk_overlap=64)


Found 4398 JSON files
Using chunking method: recursive
Processing: sunbathing-meteoroids.json
  Generated 24 chunks
Processing: de_news_events_2016_02_radical-cairo-in-shenzhen.json
  Generated 22 chunks
Processing: die-eth-zuerich-nimmt-kurs-auf-netto-null.json
  Generated 29 chunks
Processing: how-zurich-has-to-change-its-roads-to-have-more-e-bikes-than-cars.json
  Generated 49 chunks
Processing: wer-in-singapur-forschen-will-soll-sich-jetzt-melden.json
  Generated 44 chunks
Processing: recovering-hidden-treasures-and-building-boats.json
  Generated 28 chunks
Processing: responsive-ergebnisanzeige-im-wissensportal.json
  Generated 1 chunks
Processing: die-lehren-des-fernunterrichts-in-der-mathematik.json
  Generated 24 chunks
Processing: de_news_events_2019_07_herzklappen-aus-silikon.json
  Generated 35 chunks
Processing: erste-erkenntnisse-aus-der-befragung-zu-future-of-work.json
  Generated 42 chunks
Processing: der-inspirierende-blick-aus-dem-auto.json
  Generated 40 chunks
Proces

### Check chunking results

In [2]:
import chromadb
import statistics
from collections import Counter
from typing import Dict, List, Any, Generator

def connect_to_chromadb(db_path: str = "./chroma_db"):
    """Connect to ChromaDB and return client."""
    try:
        client = chromadb.PersistentClient(path=db_path)
        return client
    except Exception as e:
        print(f"Error connecting to ChromaDB: {e}")
        return None

def get_collection_basic_info(client, collection_name: str):
    """Get basic collection info without loading data."""
    try:
        collection = client.get_collection(name=collection_name)
        count = collection.count()
        return collection, count
    except Exception as e:
        print(f"Error accessing collection '{collection_name}': {e}")
        return None, 0

def analyze_collection_lightweight(collection, collection_name: str, batch_size: int = 100, max_batches: int = 10):
    """Memory-efficient analysis using small batches."""
    total_count = collection.count()
    
    if total_count == 0:
        print(f"No data found in collection '{collection_name}'")
        return
    
    print(f"\n📊 STATISTICS FOR: {collection_name}")
    print(f"Total chunks: {total_count}")
    print("-" * 40)
    
    # Initialize counters
    chunk_lengths = []
    method_counter = Counter()
    language_counter = Counter()
    filename_counter = Counter()
    
    # Process in small batches
    batches_processed = 0
    total_processed = 0
    
    try:
        # Process data in batches to avoid memory issues
        offset = 0
        while batches_processed < max_batches and offset < total_count:
            # Get small batch
            try:
                batch = collection.get(
                    limit=batch_size, 
                    offset=offset,
                    include=['documents', 'metadatas']
                )
                
                if not batch['documents']:
                    break
                
                # Process this batch
                for doc, meta in zip(batch['documents'], batch['metadatas']):
                    chunk_lengths.append(len(doc))
                    method_counter[meta.get('chunk_method', 'unknown')] += 1
                    language_counter[meta.get('language', 'unknown')] += 1
                    filename_counter[meta.get('filename', 'unknown')] += 1
                
                total_processed += len(batch['documents'])
                batches_processed += 1
                offset += batch_size
                
                # Clear batch from memory
                del batch
                
            except Exception as e:
                print(f"Error processing batch at offset {offset}: {e}")
                break
        
        if not chunk_lengths:
            print("No data retrieved for analysis")
            return
        
        print(f"Analyzed {total_processed} chunks (sample from {total_count} total)")
        
        # Calculate statistics
        print(f"\n📏 Chunk Length Statistics:")
        print(f"  Average length: {statistics.mean(chunk_lengths):.1f} characters")
        print(f"  Median length: {statistics.median(chunk_lengths):.1f} characters")
        print(f"  Min length: {min(chunk_lengths)} characters")
        print(f"  Max length: {max(chunk_lengths)} characters")
        
        # Length distribution
        length_ranges = {
            "0-100": len([l for l in chunk_lengths if 0 <= l <= 100]),
            "101-250": len([l for l in chunk_lengths if 101 <= l <= 250]),
            "251-500": len([l for l in chunk_lengths if 251 <= l <= 500]),
            "501+": len([l for l in chunk_lengths if l > 500])
        }
        
        print(f"\n📊 Length Distribution (from sample):")
        for range_name, count in length_ranges.items():
            percentage = (count / len(chunk_lengths)) * 100
            print(f"  {range_name}: {count} ({percentage:.1f}%)")
        
        # Top methods, languages, files
        print(f"\n🔧 Top Chunking Methods:")
        for method, count in method_counter.most_common(5):
            print(f"  {method}: {count}")
        
        print(f"\n🌍 Top Languages:")
        for lang, count in language_counter.most_common(5):
            print(f"  {lang}: {count}")
        
        print(f"\n📁 File Info:")
        print(f"  Unique files in sample: {len(filename_counter)}")
        if filename_counter:
            avg_chunks = total_processed / len(filename_counter)
            print(f"  Avg chunks per file: {avg_chunks:.1f}")
        
        return {
            'total_chunks': total_count,
            'sample_size': total_processed,
            'avg_length': statistics.mean(chunk_lengths),
            'methods': dict(method_counter.most_common(5)),
            'languages': dict(language_counter.most_common(5))
        }
        
    except Exception as e:
        print(f"Error during analysis: {e}")
        return None

def quick_quality_check(collection, batch_size: int = 50):
    """Quick data quality check with minimal memory usage."""
    print(f"\n🔍 QUICK QUALITY CHECK")
    print("-" * 25)
    
    try:
        # Get small sample
        sample = collection.get(limit=batch_size, include=['documents', 'metadatas'])
        
        if not sample['documents']:
            print("No data to check")
            return
        
        docs = sample['documents']
        metas = sample['metadatas']
        
        # Quick checks
        empty_chunks = len([doc for doc in docs if not doc or len(doc.strip()) == 0])
        very_short = len([doc for doc in docs if len(doc) < 10])
        missing_filename = len([meta for meta in metas if not meta.get('filename')])
        
        print(f"Sample size: {len(docs)} chunks")
        
        if empty_chunks > 0:
            print(f"⚠️  Empty chunks: {empty_chunks}")
        if very_short > 0:
            print(f"⚠️  Very short chunks (<10 chars): {very_short}")
        if missing_filename > 0:
            print(f"⚠️  Missing filename: {missing_filename}")
        
        if empty_chunks == 0 and very_short == 0 and missing_filename == 0:
            print("✅ No obvious issues in sample")
        
        # Show one sample chunk
        if docs:
            print(f"\nSample chunk ({len(docs[0])} chars):")
            print(f"  {docs[0][:80]}{'...' if len(docs[0]) > 80 else ''}")
        
    except Exception as e:
        print(f"Error in quality check: {e}")

def compare_collections_lightweight(client, collection_names: List[str]):
    """Memory-efficient collection comparison."""
    print(f"\n🔄 COLLECTION COMPARISON")
    print("="*50)
    
    comparison_data = {}
    
    for collection_name in collection_names:
        try:
            collection = client.get_collection(name=collection_name)
            count = collection.count()
            
            if count > 0:
                # Get tiny sample for quick stats
                sample = collection.get(limit=20, include=['documents'])
                if sample['documents']:
                    lengths = [len(doc) for doc in sample['documents']]
                    avg_length = statistics.mean(lengths)
                else:
                    avg_length = 0
            else:
                avg_length = 0
            
            comparison_data[collection_name] = {
                'total_chunks': count,
                'avg_length': avg_length
            }
            
        except Exception as e:
            print(f"⚠️  Error with collection '{collection_name}': {e}")
            continue
    
    if comparison_data:
        print(f"{'Collection':<30} {'Total Chunks':<12} {'Avg Length':<10}")
        print("-" * 55)
        for name, stats in comparison_data.items():
            print(f"{name:<30} {stats['total_chunks']:<12} {stats['avg_length']:<10.1f}")

def quick_verification(db_path: str = "./chroma_db", batch_size: int = 100, max_batches: int = 5):
    """Quick, memory-efficient verification."""
    print("🔍 QUICK CHROMADB VERIFICATION")
    print("="*40)
    
    # Connect
    client = connect_to_chromadb(db_path)
    if not client:
        return
    
    # List collections
    try:
        collections = client.list_collections()
        collection_names = [col.name for col in collections]
        print(f"Collections found: {len(collection_names)}")
        for name in collection_names:
            print(f"  • {name}")
    except Exception as e:
        print(f"Error listing collections: {e}")
        return
    
    if not collection_names:
        print("No collections found")
        return
    
    # Quick analysis of each collection
    for collection_name in collection_names:
        collection, count = get_collection_basic_info(client, collection_name)
        if collection and count > 0:
            analyze_collection_lightweight(collection, collection_name, batch_size, max_batches)
            quick_quality_check(collection, min(50, batch_size))
        else:
            print(f"\n📊 {collection_name}: Empty or inaccessible")
    
    # Compare if multiple collections
    if len(collection_names) > 1:
        compare_collections_lightweight(client, collection_names)
    
    print(f"\n✅ Quick verification complete!")

def check_single_collection(collection_name: str, db_path: str = "./chroma_db"):
    """Check just one collection with minimal memory usage."""
    client = connect_to_chromadb(db_path)
    if not client:
        return
    
    collection, count = get_collection_basic_info(client, collection_name)
    if collection and count > 0:
        analyze_collection_lightweight(collection, collection_name, batch_size=50, max_batches=10)
        quick_quality_check(collection)
    else:
        print(f"Collection '{collection_name}' is empty or doesn't exist")

if __name__ == "__main__":
    # Quick verification with small memory footprint
    quick_verification(batch_size=50, max_batches=5)
    
    # Or check specific collection:
    # check_single_collection("news_recursive_chunks")

🔍 QUICK CHROMADB VERIFICATION
Collections found: 2
  • news_chunks_fixed
  • news_recursive_chunks

📊 STATISTICS FOR: news_chunks_fixed
Total chunks: 1350
----------------------------------------
Analyzed 250 chunks (sample from 1350 total)

📏 Chunk Length Statistics:
  Average length: 191.9 characters
  Median length: 248.0 characters
  Min length: 10 characters
  Max length: 255 characters

📊 Length Distribution (from sample):
  0-100: 49 (19.6%)
  101-250: 110 (44.0%)
  251-500: 91 (36.4%)
  501+: 0 (0.0%)

🔧 Top Chunking Methods:
  recursive: 250

🌍 Top Languages:
  de: 127
  en: 123

📁 File Info:
  Unique files in sample: 9
  Avg chunks per file: 27.8

🔍 QUICK QUALITY CHECK
-------------------------
Sample size: 50 chunks
✅ No obvious issues in sample

Sample chunk (32 chars):
  Samples from Omani–Swiss project

📊 STATISTICS FOR: news_recursive_chunks
Total chunks: 800
----------------------------------------
Analyzed 250 chunks (sample from 800 total)

📏 Chunk Length Statistics:


### Too less chunks in crhoma db -> debug and rerun chunking

In [11]:
import json
import os
import uuid
from pathlib import Path
from typing import List, Dict, Any
import chromadb
from chromadb.config import Settings
import time

def debug_chromadb_connection(db_path: str = "./chroma_db"):
    """Debug ChromaDB connection and settings."""
    print("🔍 DEBUGGING CHROMADB CONNECTION")
    print("="*40)
    
    try:
        client = chromadb.PersistentClient(path=db_path)
        collections = client.list_collections()
        
        print(f"✅ ChromaDB connection successful")
        print(f"📁 Database path: {db_path}")
        print(f"📊 Collections found: {len(collections)}")
        
        for col in collections:
            count = col.count()
            print(f"   • {col.name}: {count} chunks")
        
        return client
    except Exception as e:
        print(f"❌ ChromaDB connection failed: {e}")
        return None

def safe_save_to_chromadb(chunks_data: List[Dict[str, Any]], collection_name: str = "news_chunks_fixed", 
                         batch_size: int = 50, db_path: str = "./chroma_db"):
    """Safely save chunks to ChromaDB with error handling and progress tracking."""
    
    if not chunks_data:
        print("No chunks to save")
        return False
    
    print(f"\n💾 SAVING {len(chunks_data)} CHUNKS TO CHROMADB")
    print("="*50)
    
    try:
        # Initialize ChromaDB client
        client = chromadb.PersistentClient(path=db_path)
        
        # Delete existing collection if it exists
        try:
            existing_collection = client.get_collection(name=collection_name)
            client.delete_collection(name=collection_name)
            print(f"🗑️  Deleted existing collection: {collection_name}")
        except:
            pass
        
        # Create new collection
        collection = client.create_collection(name=collection_name)
        print(f"✅ Created new collection: {collection_name}")
        
        # Prepare data
        documents = [chunk["text"] for chunk in chunks_data]
        metadatas = [chunk["metadata"] for chunk in chunks_data]
        ids = [chunk["id"] for chunk in chunks_data]
        
        # Ensure unique IDs
        unique_ids = []
        seen_ids = set()
        for i, chunk_id in enumerate(ids):
            if chunk_id in seen_ids:
                new_id = f"{chunk_id}_{i}"
                unique_ids.append(new_id)
            else:
                unique_ids.append(chunk_id)
                seen_ids.add(chunk_id)
        
        print(f"📊 Data prepared:")
        print(f"   • Documents: {len(documents)}")
        print(f"   • Metadata entries: {len(metadatas)}")
        print(f"   • Unique IDs: {len(unique_ids)}")
        
        # Save in smaller batches with error handling
        total_saved = 0
        failed_batches = 0
        
        for i in range(0, len(documents), batch_size):
            batch_docs = documents[i:i+batch_size]
            batch_metas = metadatas[i:i+batch_size]
            batch_ids = unique_ids[i:i+batch_size]
            
            try:
                collection.add(
                    documents=batch_docs,
                    metadatas=batch_metas,
                    ids=batch_ids
                )
                total_saved += len(batch_docs)
                
                # Progress update every 10 batches
                if (i // batch_size + 1) % 10 == 0:
                    print(f"   ✅ Saved batch {i//batch_size + 1}: {total_saved} chunks total")
                
            except Exception as e:
                failed_batches += 1
                print(f"   ❌ Failed batch {i//batch_size + 1}: {e}")
                
                # Try to save individual items from failed batch
                for j, (doc, meta, chunk_id) in enumerate(zip(batch_docs, batch_metas, batch_ids)):
                    try:
                        collection.add(
                            documents=[doc],
                            metadatas=[meta],
                            ids=[chunk_id]
                        )
                        total_saved += 1
                    except Exception as individual_error:
                        print(f"     ❌ Failed individual item {j}: {individual_error}")
            
            # Small delay to prevent overwhelming ChromaDB
            time.sleep(0.01)
        
        # Verify final count
        final_count = collection.count()
        
        print(f"\n📊 SAVE RESULTS:")
        print(f"   • Total chunks processed: {len(chunks_data)}")
        print(f"   • Successfully saved: {total_saved}")
        print(f"   • Failed batches: {failed_batches}")
        print(f"   • Final ChromaDB count: {final_count}")
        print(f"   • Success rate: {(final_count/len(chunks_data))*100:.1f}%")
        
        if final_count == len(chunks_data):
            print("✅ All chunks saved successfully!")
        else:
            print(f"⚠️  {len(chunks_data) - final_count} chunks missing")
        
        return final_count == len(chunks_data)
        
    except Exception as e:
        print(f"❌ Critical error during save: {e}")
        return False

def diagnose_existing_chunks(collection_name: str = "news_recursive_chunks", db_path: str = "./chroma_db"):
    """Diagnose what happened with existing chunks."""
    print(f"\n🔍 DIAGNOSING EXISTING COLLECTION: {collection_name}")
    print("="*50)
    
    try:
        client = chromadb.PersistentClient(path=db_path)
        collection = client.get_collection(name=collection_name)
        
        total_count = collection.count()
        print(f"📊 Current count: {total_count}")
        
        if total_count > 0:
            # Get sample to analyze
            sample = collection.get(limit=min(50, total_count), include=['documents', 'metadatas'])
            
            docs = sample['documents']
            metas = sample['metadatas']
            
            # Check for patterns in existing data
            filenames = set([meta.get('filename', 'unknown') for meta in metas])
            methods = set([meta.get('chunk_method', 'unknown') for meta in metas])
            
            print(f"📁 Unique files represented: {len(filenames)}")
            print(f"🔧 Chunking methods: {methods}")
            
            # Show some filenames
            print(f"📋 Sample filenames:")
            for i, filename in enumerate(list(filenames)[:5]):
                print(f"   • {filename}")
            
            # Check if data looks truncated
            if total_count < 1000:
                print(f"⚠️  Low chunk count suggests saving was interrupted")
                
                # Look for pattern in last saved items
                last_batch = collection.get(
                    limit=10, 
                    offset=max(0, total_count-10),
                    include=['metadatas']
                )
                
                if last_batch['metadatas']:
                    last_files = [meta.get('filename', 'unknown') for meta in last_batch['metadatas']]
                    print(f"📄 Last files saved: {set(last_files)}")
        
        return total_count
        
    except Exception as e:
        print(f"❌ Error diagnosing collection: {e}")
        return 0

def reprocess_and_save_safely(json_directory: str, chunking_method: str = "recursive", 
                             collection_name: str = "news_chunks_fixed", **kwargs):
    """Reprocess JSON files and save safely to ChromaDB."""
    
    print(f"\n🔄 REPROCESSING AND SAVING SAFELY")
    print("="*50)
    
    # Import chunking functions (assuming they're available)
    #from text_chunking_script import process_json_file  # Adjust import as needed
    
    directory = Path(json_directory)
    json_files = list(directory.rglob("*.json"))
    
    if not json_files:
        print(f"No JSON files found in {json_directory}")
        return
    
    print(f"📁 Found {len(json_files)} JSON files")
    
    all_chunks = []
    processed_files = 0
    
    for json_file in json_files:
        try:
            chunks = process_json_file(str(json_file), chunking_method, **kwargs)
            all_chunks.extend(chunks)
            processed_files += 1
            
            # Progress update
            if processed_files % 100 == 0:
                print(f"   📊 Processed {processed_files} files, {len(all_chunks)} chunks total")
                
        except Exception as e:
            print(f"❌ Error processing {json_file}: {e}")
    
    print(f"\n📊 PROCESSING COMPLETE:")
    print(f"   • Files processed: {processed_files}")
    print(f"   • Total chunks generated: {len(all_chunks)}")
    
    if all_chunks:
        # Save safely
        success = safe_save_to_chromadb(all_chunks, collection_name)
        return success
    else:
        print("❌ No chunks generated")
        return False

def main():
    """Main diagnostic and fix function."""
    # Step 1: Debug connection
    client = debug_chromadb_connection()
    if not client:
        return
    
    # Step 2: Diagnose existing collection
    existing_count = diagnose_existing_chunks()
    
    # Step 3: Offer to reprocess if needed
    if existing_count < 50000:  # Assuming you expect many more chunks
        print(f"\n🔧 RECOMMENDATION: Reprocess files with safe saving")
        print(f"   Current count ({existing_count}) seems too low")
        
        # You can uncomment and modify this to reprocess:
        # json_directory = "/path/to/your/json/files"
        # reprocess_and_save_safely(
        #     json_directory=json_directory,
        #     chunking_method="recursive",
        #     collection_name="news_chunks_fixed",
        #     chunk_size=256,
        #     chunk_overlap=64
        # )

if __name__ == "__main__":
    main()

🔍 DEBUGGING CHROMADB CONNECTION
✅ ChromaDB connection successful
📁 Database path: ./chroma_db
📊 Collections found: 0

🔍 DIAGNOSING EXISTING COLLECTION: news_recursive_chunks
❌ Error diagnosing collection: Collection [news_recursive_chunks] does not exists

🔧 RECOMMENDATION: Reprocess files with safe saving
   Current count (0) seems too low


In [12]:
# Update the path in the script
json_directory = "/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2"

reprocess_and_save_safely(
    json_directory=json_directory,
    chunking_method="recursive", 
    collection_name="news_chunks_recursive",
    chunk_size=256,
    chunk_overlap=64
)


🔄 REPROCESSING AND SAVING SAFELY
📁 Found 4398 JSON files
   📊 Processed 100 files, 2588 chunks total
   📊 Processed 200 files, 4726 chunks total
   📊 Processed 300 files, 7240 chunks total
   📊 Processed 400 files, 9870 chunks total
   📊 Processed 500 files, 12290 chunks total
   📊 Processed 600 files, 14774 chunks total
   📊 Processed 700 files, 17438 chunks total
   📊 Processed 800 files, 19802 chunks total
   📊 Processed 900 files, 22278 chunks total
   📊 Processed 1000 files, 24940 chunks total
   📊 Processed 1100 files, 27385 chunks total
   📊 Processed 1200 files, 29905 chunks total
   📊 Processed 1300 files, 32455 chunks total
   📊 Processed 1400 files, 34909 chunks total
   📊 Processed 1500 files, 37265 chunks total
   📊 Processed 1600 files, 39922 chunks total
   📊 Processed 1700 files, 42609 chunks total
   📊 Processed 1800 files, 45005 chunks total
   📊 Processed 1900 files, 47412 chunks total
   📊 Processed 2000 files, 49907 chunks total
   📊 Processed 2100 files, 52258 ch

True

### Delete collection

In [13]:
import chromadb

def delete_collection(collection_name: str, db_path: str = "./chroma_db"):
    """Delete a ChromaDB collection."""
    try:
        client = chromadb.PersistentClient(path=db_path)
        client.delete_collection(name=collection_name)
        print(f"✅ Deleted collection: {collection_name}")
    except Exception as e:
        print(f"⚠️  Collection '{collection_name}' not found or already deleted: {e}")

# Delete the collection
delete_collection("news_chunks_fixed")

✅ Deleted collection: news_chunks_fixed


### check existing collections and create backups

In [7]:
import chromadb
import json
import shutil
from pathlib import Path
from datetime import datetime

# =============================================================================
# 1. BASIC PERSISTENCE (Already working!)
# =============================================================================

def save_and_access_chromadb():
    """Your ChromaDB is already persistent! Here's how to access it."""
    
    # When you use PersistentClient, data is automatically saved to disk
    db_path = "./chroma_db"  # This is your database directory
    
    # Session 1: Create and save data
    print("📁 Session 1: Creating collection...")
    client = chromadb.PersistentClient(path=db_path)
    
    # Your collection is automatically saved when you add data
    # No additional save step needed!
    
    # Session 2: Access the same data (in a new Python session)
    print("📂 Session 2: Accessing existing collection...")
    client = chromadb.PersistentClient(path=db_path)  # Same path!
    
    # List existing collections
    collections = client.list_collections()
    for col in collections:
        print(f"   Found collection: {col.name} with {col.count()} chunks")
    
    return client

# =============================================================================
# 2. VERIFY PERSISTENCE
# =============================================================================

def verify_persistence(db_path: str = "./chroma_db"):
    """Verify that your data persists across sessions."""
    print(f"🔍 VERIFYING PERSISTENCE")
    print("="*30)
    
    # Check if database directory exists
    if Path(db_path).exists():
        print(f"✅ Database directory exists: {db_path}")
        
        # List files in the directory
        files = list(Path(db_path).rglob("*"))
        print(f"📁 Database contains {len(files)} files")
        
        # Connect and check collections
        try:
            client = chromadb.PersistentClient(path=db_path)
            collections = client.list_collections()
            
            print(f"📊 Collections found: {len(collections)}")
            for col in collections:
                count = col.count()
                print(f"   • {col.name}: {count:,} chunks")
            
            return True
        except Exception as e:
            print(f"❌ Error accessing database: {e}")
            return False
    else:
        print(f"❌ Database directory not found: {db_path}")
        return False

# =============================================================================
# 3. BACKUP YOUR DATABASE
# =============================================================================

def backup_chromadb(source_path: str = "./chroma_db", backup_dir: str = "./chromadb_backups"):
    """Create a backup of your ChromaDB."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = Path(backup_dir) / f"chromadb_backup_{timestamp}"
    
    try:
        # Create backup directory
        backup_path.mkdir(parents=True, exist_ok=True)
        
        # Copy entire database directory
        if Path(source_path).exists():
            shutil.copytree(source_path, backup_path / "chroma_db")
            print(f"✅ Backup created: {backup_path}")
            return str(backup_path)
        else:
            print(f"❌ Source database not found: {source_path}")
            return None
    except Exception as e:
        print(f"❌ Backup failed: {e}")
        return None

def restore_chromadb(backup_path: str, target_path: str = "./chroma_db_restored"):
    """Restore ChromaDB from backup."""
    try:
        backup_db = Path(backup_path) / "chroma_db"
        if backup_db.exists():
            if Path(target_path).exists():
                shutil.rmtree(target_path)
            shutil.copytree(backup_db, target_path)
            print(f"✅ Database restored to: {target_path}")
            return True
        else:
            print(f"❌ Backup not found: {backup_db}")
            return False
    except Exception as e:
        print(f"❌ Restore failed: {e}")
        return False

# =============================================================================
# 4. EXPORT COLLECTION DATA
# =============================================================================

def export_collection_to_json(collection_name: str, output_file: str, db_path: str = "./chroma_db"):
    """Export a collection to JSON file for additional backup."""
    try:
        client = chromadb.PersistentClient(path=db_path)
        collection = client.get_collection(name=collection_name)
        
        # Get all data
        total_count = collection.count()
        batch_size = 1000
        all_data = []
        
        print(f"📤 Exporting {total_count} chunks from '{collection_name}'...")
        
        for offset in range(0, total_count, batch_size):
            batch = collection.get(
                limit=batch_size,
                offset=offset,
                include=['documents', 'metadatas', 'ids']
            )
            
            for doc, meta, chunk_id in zip(batch['documents'], batch['metadatas'], batch['ids']):
                all_data.append({
                    'id': chunk_id,
                    'document': doc,
                    'metadata': meta
                })
            
            print(f"   Exported {min(offset + batch_size, total_count)} / {total_count}")
        
        # Save to JSON
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(all_data, f, ensure_ascii=False, indent=2)
        
        print(f"✅ Collection exported to: {output_file}")
        return True
        
    except Exception as e:
        print(f"❌ Export failed: {e}")
        return False

# =============================================================================
# 5. SIMPLE ACCESS PATTERN FOR NEW SESSIONS
# =============================================================================

def access_chromadb_new_session(db_path: str = "./chroma_db"):
    """Simple function to access ChromaDB in any new session."""
    try:
        # Connect to existing database
        client = chromadb.PersistentClient(path=db_path)
        
        # List available collections
        collections = client.list_collections()
        print(f"📊 Available collections:")
        
        collection_dict = {}
        for col in collections:
            count = col.count()
            collection_dict[col.name] = col
            print(f"   • {col.name}: {count:,} chunks")
        
        return client, collection_dict
        
    except Exception as e:
        print(f"❌ Error accessing ChromaDB: {e}")
        return None, {}

# =============================================================================
# USAGE EXAMPLES
# =============================================================================

if __name__ == "__main__":
    # Check if your database persists
    verify_persistence()
    
    # Access collections in a new session
    # client, collections = access_chromadb_new_session()
    
    # Create a backup (recommended!)
    backup_path = backup_chromadb()
    
    # Export a specific collection to JSON
    #if collections:
    #    collection_name = list(collections.keys())[0]  # First collection
    #    export_collection_to_json(collection_name, f"{collection_name}_export.json")

# =============================================================================
# QUICK ACCESS FOR NEW SESSIONS
# =============================================================================

"""
For any new Python session, just use:

import chromadb

# Connect to your existing database
client = chromadb.PersistentClient(path="./chroma_db")

# Get your collection
collection = client.get_collection("news_chunks_fixed")

# Use it normally
results = collection.query(
    query_texts=["your search text"],
    n_results=5
)
"""

🔍 VERIFYING PERSISTENCE
✅ Database directory exists: ./chroma_db
📁 Database contains 36 files
📊 Collections found: 2
   • news_structure_chunks: 56,977 chunks
   • news_recursive_chunks: 109,128 chunks
✅ Backup created: chromadb_backups/chromadb_backup_20250523_230726


'\nFor any new Python session, just use:\n\nimport chromadb\n\n# Connect to your existing database\nclient = chromadb.PersistentClient(path="./chroma_db")\n\n# Get your collection\ncollection = client.get_collection("news_chunks_fixed")\n\n# Use it normally\nresults = collection.query(\n    query_texts=["your search text"],\n    n_results=5\n)\n'

### Create further collections with different chunking methods

In [ ]:
%pip install tf-keras

In [ ]:
process_directory(json_directory, "semantic", "news_semantic_chunks", model_name="mini")


In [16]:
process_directory(json_directory, "document_structure", "news_structure_chunks")


Found 4398 JSON files
Using chunking method: document_structure
Processing: sunbathing-meteoroids.json
  Generated 10 chunks
Processing: de_news_events_2016_02_radical-cairo-in-shenzhen.json
  Generated 10 chunks
Processing: die-eth-zuerich-nimmt-kurs-auf-netto-null.json
  Generated 15 chunks
Processing: how-zurich-has-to-change-its-roads-to-have-more-e-bikes-than-cars.json
  Generated 22 chunks
Processing: wer-in-singapur-forschen-will-soll-sich-jetzt-melden.json
  Generated 27 chunks
Processing: recovering-hidden-treasures-and-building-boats.json
  Generated 13 chunks
Processing: responsive-ergebnisanzeige-im-wissensportal.json
  Generated 1 chunks
Processing: die-lehren-des-fernunterrichts-in-der-mathematik.json
  Generated 9 chunks
Processing: de_news_events_2019_07_herzklappen-aus-silikon.json
  Generated 20 chunks
Processing: erste-erkenntnisse-aus-der-befragung-zu-future-of-work.json
  Generated 16 chunks
Processing: der-inspirierende-blick-aus-dem-auto.json
  Generated 16 chunk

In [ ]:
import time
from langchain_huggingface import HuggingFaceEmbeddings

def get_embedding_models_safe():
    """Initialize embedding models with rate limiting protection."""
    model_kwargs = {"device": "cpu"}
    
    # Only initialize the model you actually use
    embedding_models = {}
    
    # Add delay between model loads to avoid rate limiting
    print("🔄 Loading embedding model safely...")
    
    try:
        # Only load mini model first (most reliable)
        embedding_models["mini"] = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2", 
            model_kwargs=model_kwargs
        )
        print("✅ Loaded 'mini' model successfully")
        time.sleep(2)  # Delay to avoid rate limiting
        
    except Exception as e:
        print(f"❌ Failed to load 'mini' model: {e}")
    
    return embedding_models

def chunk_paragraph_semantic_safe(paragraph: str, model_name: str = "mini") -> List[str]:
    """Semantic chunking with better error handling."""
    if not paragraph or len(paragraph.strip()) == 0:
        return []
    
    try:
        # Use cached models or initialize safely
        if not hasattr(chunk_paragraph_semantic_safe, 'cached_models'):
            chunk_paragraph_semantic_safe.cached_models = get_embedding_models_safe()
        
        embedding_models = chunk_paragraph_semantic_safe.cached_models
        
        if model_name not in embedding_models:
            print(f"Model {model_name} not available, falling back to recursive chunking")
            return chunk_paragraph_recursive(paragraph)
        
        from langchain_experimental.text_splitter import SemanticChunker
        
        semantic_chunker = SemanticChunker(
            embeddings=embedding_models[model_name],
            breakpoint_threshold_type="percentile"
        )
        
        chunks = semantic_chunker.split_text(paragraph)
        return [chunk.strip() for chunk in chunks if chunk.strip()]
        
    except Exception as e:
        print(f"Semantic chunking failed, using recursive fallback: {e}")
        return chunk_paragraph_recursive(paragraph)

# Quick fix: Use only recursive chunking to avoid the issue entirely
def process_directory_safe(directory_path: str, collection_name: str = "news_chunks_safe"):
    """Process directory using only recursive chunking (no HuggingFace downloads)."""
    from pathlib import Path
    
    directory = Path(directory_path)
    json_files = list(directory.rglob("*.json"))
    
    print(f"📁 Found {len(json_files)} JSON files")
    print(f"🔧 Using recursive chunking only (avoiding rate limits)")
    
    all_chunks = []
    
    for json_file in json_files:
        print(f"Processing: {json_file.name}")
        try:
            chunks = process_json_file(str(json_file), "recursive", chunk_size=256, chunk_overlap=64)
            all_chunks.extend(chunks)
            print(f"  Generated {len(chunks)} chunks")
        except Exception as e:
            print(f"  Error: {e}")
    
    print(f"\nTotal chunks: {len(all_chunks)}")
    
    # Save to ChromaDB
    if all_chunks:
        from chromadb_debug_script import safe_save_to_chromadb
        safe_save_to_chromadb(all_chunks, collection_name)
    
    return len(all_chunks)

# Alternative: Cache models locally first
def download_models_locally():
    """Download and cache models locally to avoid repeated requests."""
    print("📥 Downloading models locally to avoid rate limits...")
    
    try:
        # Download mini model
        model_kwargs = {"device": "cpu"}
        model = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2", 
            model_kwargs=model_kwargs
        )
        print("✅ Mini model cached locally")
        
        # Test it works
        test_embedding = model.embed_query("test")
        print(f"✅ Model working, embedding size: {len(test_embedding)}")
        
        return True
        
    except Exception as e:
        print(f"❌ Failed to cache model: {e}")
        return False

if __name__ == "__main__":
    # Option 1: Download model first, then use semantic chunking
    print("Option 1: Cache model locally first")
    if download_models_locally():
        print("✅ Now you can use semantic chunking safely")
    
    print("\nOption 2: Use safe recursive chunking only")
    # json_directory = "/your/path"
    # process_directory_safe(json_directory, "news_chunks_no_rate_limit")

In [ ]:
## Check how many collections exist

In [15]:
import chromadb
from pathlib import Path

def check_chromadb_counts(db_path: str = "./chroma_db"):
    """Simple check of ChromaDB chunk counts."""
    print("🔍 CHECKING CHROMADB CHUNKS")
    print("="*30)
    
    try:
        # Connect to ChromaDB
        client = chromadb.PersistentClient(path=db_path)
        
        # List all collections
        collections = client.list_collections()
        
        if not collections:
            print("❌ No collections found")
            return
        
        print(f"📊 Found {len(collections)} collection(s):")
        print()
        
        total_chunks = 0
        
        for collection in collections:
            count = collection.count()
            total_chunks += count
            
            print(f"📁 {collection.name}")
            print(f"   Chunks: {count:,}")
            
            # Get one sample chunk to verify data exists
            if count > 0:
                try:
                    sample = collection.get(limit=1, include=['documents', 'metadatas'])
                    if sample['documents']:
                        doc_length = len(sample['documents'][0])
                        filename = sample['metadatas'][0].get('filename', 'unknown')
                        print(f"   Sample: {doc_length} chars from '{filename}'")
                except:
                    print(f"   ⚠️  Error reading sample data")
            print()
        
        print(f"📈 TOTAL CHUNKS ACROSS ALL COLLECTIONS: {total_chunks:,}")
        
        # Quick success check
        if total_chunks > 50000:
            print("✅ Looks good! Large number of chunks found")
        elif total_chunks > 1000:
            print("🟡 Moderate number of chunks - might be partial")
        else:
            print("❌ Low chunk count - possible issue")
            
    except Exception as e:
        print(f"❌ Error checking ChromaDB: {e}")

def check_specific_collection(collection_name: str, db_path: str = "./chroma_db"):
    """Check a specific collection."""
    print(f"🔍 CHECKING COLLECTION: {collection_name}")
    print("="*40)
    
    try:
        client = chromadb.PersistentClient(path=db_path)
        collection = client.get_collection(name=collection_name)
        
        count = collection.count()
        print(f"📊 Total chunks: {count:,}")
        
        if count > 0:
            # Get sample
            sample = collection.get(limit=3, include=['documents', 'metadatas'])
            
            print(f"\n📝 Sample chunks:")
            for i, (doc, meta) in enumerate(zip(sample['documents'], sample['metadatas'])):
                filename = meta.get('filename', 'unknown')
                method = meta.get('chunk_method', 'unknown')
                print(f"   {i+1}. {filename} ({method}) - {len(doc)} chars")
                print(f"      {doc[:60]}{'...' if len(doc) > 60 else ''}")
        
        return count
        
    except Exception as e:
        print(f"❌ Collection '{collection_name}' not found or error: {e}")
        return 0

def quick_status():
    """Super quick status check."""
    try:
        client = chromadb.PersistentClient(path="./chroma_db")
        collections = client.list_collections()
        
        print("📊 QUICK STATUS:")
        for col in collections:
            count = col.count()
            print(f"   {col.name}: {count:,} chunks")
            
    except Exception as e:
        print(f"❌ Error: {e}")

if __name__ == "__main__":
    # Quick check of all collections
    check_chromadb_counts()
    
    # Or check specific collection:
    check_specific_collection("news_structure_chunks") #news_chunks_recursive news_structure_chunks
    
    # Or super quick status:
    # quick_status()

🔍 CHECKING CHROMADB CHUNKS
📊 Found 2 collection(s):

📁 news_structure_chunks
   Chunks: 56,977
   Sample: 480 chars from 'sunbathing-meteoroids.md'

📁 news_recursive_chunks
   Chunks: 109,128
   Sample: 32 chars from 'sunbathing-meteoroids.md'

📈 TOTAL CHUNKS ACROSS ALL COLLECTIONS: 166,105
✅ Looks good! Large number of chunks found
🔍 CHECKING COLLECTION: news_structure_chunks
📊 Total chunks: 56,977

📝 Sample chunks:
   1. sunbathing-meteoroids.md (document_structure) - 480 chars
      Samples from Omani–Swiss project
The inconspicuous, small st...
   2. sunbathing-meteoroids.md (document_structure) - 238 chars
      Then, after a rapid transfer from the asteroid belt to Earth...
   3. sunbathing-meteoroids.md (document_structure) - 480 chars
      Photomicrograph of Jiddat al Harasis 466 (thin-section). (Im...


In [ ]:
import chromadb

def check_both_collections():
    """Check if both chunking collections exist."""
    print("🔍 CHECKING MULTIPLE COLLECTIONS")
    print("="*35)
    
    try:
        client = chromadb.PersistentClient(path="./chroma_db")
        
        # Check for specific collections
        collections_to_check = [
            "news_hybrid_chunks",
            "news_structure_chunks",
            "news_chunks_fixed",  # Your previous one
            "news_recursive_chunks"  # If this exists
        ]
        
        found_collections = []
        
        for collection_name in collections_to_check:
            try:
                collection = client.get_collection(name=collection_name)
                count = collection.count()
                found_collections.append((collection_name, count))
                print(f"✅ {collection_name}: {count:,} chunks")
            except:
                print(f"❌ {collection_name}: Not found")
        
        print(f"\n📊 SUMMARY:")
        print(f"   Found {len(found_collections)} collections")
        total_chunks = sum(count for _, count in found_collections)
        print(f"   Total chunks across all: {total_chunks:,}")
        
        # Show all collections in database
        all_collections = client.list_collections()
        print(f"\n📁 ALL COLLECTIONS IN DATABASE:")
        for col in all_collections:
            count = col.count()
            print(f"   • {col.name}: {count:,} chunks")
        
        return found_collections
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return []

def compare_chunking_methods():
    """Compare different chunking methods."""
    print("\n🔄 COMPARING CHUNKING METHODS")
    print("="*30)
    
    try:
        client = chromadb.PersistentClient(path="./chroma_db")
        
        methods = {
            "news_hybrid_chunks": "Hybrid (Semantic + Recursive)",
            "news_structure_chunks": "Document Structure", 
            "news_chunks_fixed": "Recursive"
        }
        
        for collection_name, method_name in methods.items():
            try:
                collection = client.get_collection(name=collection_name)
                count = collection.count()
                
                # Get sample to check average length
                if count > 0:
                    sample = collection.get(limit=10, include=['documents'])
                    if sample['documents']:
                        lengths = [len(doc) for doc in sample['documents']]
                        avg_length = sum(lengths) / len(lengths)
                        print(f"📊 {method_name}:")
                        print(f"   Chunks: {count:,}")
                        print(f"   Avg length: {avg_length:.1f} chars")
                        print()
                
            except:
                print(f"❌ {method_name}: Collection not found")
        
    except Exception as e:
        print(f"❌ Error comparing methods: {e}")

if __name__ == "__main__":
    # Check if both collections exist
    found = check_both_collections()
    
    # Compare the different methods
    if len(found) > 1:
        compare_chunking_methods()
    
    print("✅ Check complete!")

In [16]:
# =============================================================================
# USAGE EXAMPLES
# =============================================================================

if __name__ == "__main__":
    # Check if your database persists
    verify_persistence()
    
    # Access collections in a new session
    # client, collections = access_chromadb_new_session()
    
    # Create a backup (recommended!)
    backup_path = backup_chromadb()
    
    # Export a specific collection to JSON
    #if collections:
    #    collection_name = list(collections.keys())[0]  # First collection
    #    export_collection_to_json(collection_name, f"{collection_name}_export.json")

# =============================================================================
# QUICK ACCESS FOR NEW SESSIONS
# =============================================================================

"""
For any new Python session, just use:

import chromadb

# Connect to your existing database
client = chromadb.PersistentClient(path="./chroma_db")

# Get your collection
collection = client.get_collection("news_chunks_fixed")

# Use it normally
results = collection.query(
    query_texts=["your search text"],
    n_results=5
)
"""

🔍 VERIFYING PERSISTENCE
✅ Database directory exists: ./chroma_db
📁 Database contains 36 files
📊 Collections found: 2
   • news_structure_chunks: 56,977 chunks
   • news_recursive_chunks: 109,128 chunks
✅ Backup created: chromadb_backups/chromadb_backup_20250524_220135


'\nFor any new Python session, just use:\n\nimport chromadb\n\n# Connect to your existing database\nclient = chromadb.PersistentClient(path="./chroma_db")\n\n# Get your collection\ncollection = client.get_collection("news_chunks_fixed")\n\n# Use it normally\nresults = collection.query(\n    query_texts=["your search text"],\n    n_results=5\n)\n'

In [29]:
import chromadb

# Same path = same data
client = chromadb.PersistentClient(path="./chroma_db")

# Get your collection
collection = client.get_collection("news_structure_chunks")

# Use normally
results = collection.query(query_texts=["brain research"], n_results=5)